# 🛰️ AIOps IoT Monitoring — Colab Demo
**Project 2: AIOps Enabled Secure IoT & Electronics Monitoring System**

Self-contained notebook. No external files needed.

**Steps:**
1. Install dependencies
2. Unpack source files (base64 embedded pattern)
3. Seed SQLite DB with 12 devices × 100 readings
4. Train IsolationForest anomaly detector
5. Detect anomalies → write alerts
6. Show identity-aware agent tokens (7 agents)
7. Generate QR code for a device
8. Generate + download PDF incident report
9. Final summary


In [ ]:
# Cell 1: Install dependencies
!pip install sqlalchemy python-dotenv scikit-learn pandas numpy joblib pyjwt bcrypt 'qrcode[pil]' reportlab --quiet
print('Dependencies installed')

In [ ]:
# Cell 2: Unpack embedded source files
import os, base64, json
BUNDLE = json.loads('{"backend/__init__.py": "", "backend/db/__init__.py": "", "backend/simulator/__init__.py": "", "backend/ml/__init__.py": "", "backend/auth/__init__.py": "", "backend/db/models.py": "IiIiCkRhdGFiYXNlIFNjaGVtYSAtIEFJT3BzIElvVCBNb25pdG9yaW5nCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQo1IHRhYmxlcyBtYXRjaGluZyB0aGUgYXNzaWdubWVudCBzcGVjaWZpY2F0aW9uOgoxLiBkZXZpY2VzIC0gSW9UIGRldmljZSByZWdpc3RyeQoyLiBzZW5zb3JfcmVhZGluZ3MgLSBlbnZpcm9ubWVudGFsIHRlbGVtZXRyeQozLiBhbGVydHMgLSBhbm9tYWx5IGZsYWdzICsgbWFpbnRlbmFuY2UgcmVjb21tZW5kYXRpb25zCjQuIGluY2lkZW50cyAtIGNsYXNzaWZpZWQgdGhyZWF0cyB3aXRoIHJlc3BvbnNlIHJlY29tbWVuZGF0aW9ucwo1LiBhZ2VudF90YXNrX2xvZ3MgLSBpZGVudGl0eS1hd2FyZSBhZ2VudCBkZWNpc2lvbiBhdWRpdCB0cmFpbAoiIiIKCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gc3FsYWxjaGVteSBpbXBvcnQgKAogICAgQ29sdW1uLCBJbnRlZ2VyLCBTdHJpbmcsIEZsb2F0LCBCb29sZWFuLCBEYXRlVGltZSwgRm9yZWlnbktleSwgVGV4dCwgY3JlYXRlX2VuZ2luZQopCmZyb20gc3FsYWxjaGVteS5vcm0gaW1wb3J0IGRlY2xhcmF0aXZlX2Jhc2UsIHJlbGF0aW9uc2hpcCwgc2Vzc2lvbm1ha2VyCmltcG9ydCBvcwpmcm9tIGRvdGVudiBpbXBvcnQgbG9hZF9kb3RlbnYKCmxvYWRfZG90ZW52KCkKCkRBVEFCQVNFX1VSTCA9IG9zLmdldGVudigiREFUQUJBU0VfVVJMIiwgInNxbGl0ZTovLy8uL2Fpb3BzX2lvdC5kYiIpCgplbmdpbmUgPSBjcmVhdGVfZW5naW5lKAogICAgREFUQUJBU0VfVVJMLAogICAgY29ubmVjdF9hcmdzPXsiY2hlY2tfc2FtZV90aHJlYWQiOiBGYWxzZX0gaWYgInNxbGl0ZSIgaW4gREFUQUJBU0VfVVJMIGVsc2Uge30KKQpTZXNzaW9uTG9jYWwgPSBzZXNzaW9ubWFrZXIoYXV0b2NvbW1pdD1GYWxzZSwgYXV0b2ZsdXNoPUZhbHNlLCBiaW5kPWVuZ2luZSkKQmFzZSA9IGRlY2xhcmF0aXZlX2Jhc2UoKQoKCmNsYXNzIERldmljZShCYXNlKToKICAgICIiIklvVCBkZXZpY2UgcmVnaXN0cnkgLSBzZW5zb3JzIGRlcGxveWVkIGluIHdhcmVob3VzZS4iIiIKICAgIF9fdGFibGVuYW1lX18gPSAiZGV2aWNlcyIKCiAgICBpZCA9IENvbHVtbihJbnRlZ2VyLCBwcmltYXJ5X2tleT1UcnVlLCBpbmRleD1UcnVlKQogICAgZGV2aWNlX2lkID0gQ29sdW1uKFN0cmluZyg1MCksIHVuaXF1ZT1UcnVlLCBudWxsYWJsZT1GYWxzZSwgaW5kZXg9VHJ1ZSkKICAgIG5hbWUgPSBDb2x1bW4oU3RyaW5nKDEwMCksIG51bGxhYmxlPUZhbHNlKQogICAgbG9jYXRpb24gPSBDb2x1bW4oU3RyaW5nKDEwMCksIG51bGxhYmxlPUZhbHNlKSAgIyBlLmcuLCAiV2FyZWhvdXNlLUEtWm9uZS0zIgogICAgZGV2aWNlX3R5cGUgPSBDb2x1bW4oU3RyaW5nKDUwKSwgbnVsbGFibGU9RmFsc2UpICAjIHRlbXBfc2Vuc29yLCBodW1pZGl0eV9zZW5zb3IsIHZpYnJhdGlvbl9zZW5zb3IKICAgIGZpcm13YXJlX3ZlcnNpb24gPSBDb2x1bW4oU3RyaW5nKDIwKSwgbnVsbGFibGU9RmFsc2UpCiAgICBhdXRoX3N0YXR1cyA9IENvbHVtbihTdHJpbmcoMjApLCBkZWZhdWx0PSJhdXRoZW50aWNhdGVkIikgICMgYXV0aGVudGljYXRlZCwgdW5hdXRob3JpemVkLCBzdXNwaWNpb3VzCiAgICBiYXR0ZXJ5X2xldmVsID0gQ29sdW1uKEZsb2F0LCBkZWZhdWx0PTEwMC4wKQogICAgaXNfYWN0aXZlID0gQ29sdW1uKEJvb2xlYW4sIGRlZmF1bHQ9VHJ1ZSkKICAgIHJlZ2lzdGVyZWRfYXQgPSBDb2x1bW4oRGF0ZVRpbWUsIGRlZmF1bHQ9ZGF0ZXRpbWUudXRjbm93KQogICAgbGFzdF9zZWVuID0gQ29sdW1uKERhdGVUaW1lLCBkZWZhdWx0PWRhdGV0aW1lLnV0Y25vdykKCiAgICByZWFkaW5ncyA9IHJlbGF0aW9uc2hpcCgiU2Vuc29yUmVhZGluZyIsIGJhY2tfcG9wdWxhdGVzPSJkZXZpY2UiLCBjYXNjYWRlPSJhbGwsIGRlbGV0ZS1vcnBoYW4iKQogICAgYWxlcnRzID0gcmVsYXRpb25zaGlwKCJBbGVydCIsIGJhY2tfcG9wdWxhdGVzPSJkZXZpY2UiLCBjYXNjYWRlPSJhbGwsIGRlbGV0ZS1vcnBoYW4iKQogICAgaW5jaWRlbnRzID0gcmVsYXRpb25zaGlwKCJJbmNpZGVudCIsIGJhY2tfcG9wdWxhdGVzPSJkZXZpY2UiLCBjYXNjYWRlPSJhbGwsIGRlbGV0ZS1vcnBoYW4iKQoKCmNsYXNzIFNlbnNvclJlYWRpbmcoQmFzZSk6CiAgICAiIiJFbnZpcm9ubWVudGFsIHRlbGVtZXRyeSBmcm9tIGRldmljZXMuIiIiCiAgICBfX3RhYmxlbmFtZV9fID0gInNlbnNvcl9yZWFkaW5ncyIKCiAgICBpZCA9IENvbHVtbihJbnRlZ2VyLCBwcmltYXJ5X2tleT1UcnVlLCBpbmRleD1UcnVlKQogICAgZGV2aWNlX2lkID0gQ29sdW1uKFN0cmluZyg1MCksIEZvcmVpZ25LZXkoImRldmljZXMuZGV2aWNlX2lkIiksIG51bGxhYmxlPUZhbHNlLCBpbmRleD1UcnVlKQogICAgdGltZXN0YW1wID0gQ29sdW1uKERhdGVUaW1lLCBkZWZhdWx0PWRhdGV0aW1lLnV0Y25vdywgaW5kZXg9VHJ1ZSkKICAgIHRlbXBlcmF0dXJlID0gQ29sdW1uKEZsb2F0KSAgIyBDZWxzaXVzCiAgICBodW1pZGl0eSA9IENvbHVtbihGbG9hdCkgICAgICMgJQogICAgdmlicmF0aW9uID0gQ29sdW1uKEZsb2F0KSAgICAjIG1tL3MgUk1TCiAgICBiYXR0ZXJ5ID0gQ29sdW1uKEZsb2F0KSAgICAgICMgJQogICAgc2lnbmFsX3N0cmVuZ3RoID0gQ29sdW1uKEZsb2F0KSAgIyBkQm0KCiAgICBkZXZpY2UgPSByZWxhdGlvbnNoaXAoIkRldmljZSIsIGJhY2tfcG9wdWxhdGVzPSJyZWFkaW5ncyIpCgoKY2xhc3MgQWxlcnQoQmFzZSk6CiAgICAiIiJBbm9tYWx5IGZsYWdzIHJhaXNlZCBieSBkZXRlY3Rpb24gYWdlbnRzLiIiIgogICAgX190YWJsZW5hbWVfXyA9ICJhbGVydHMiCgogICAgaWQgPSBDb2x1bW4oSW50ZWdlciwgcHJpbWFyeV9rZXk9VHJ1ZSwgaW5kZXg9VHJ1ZSkKICAgIGRldmljZV9pZCA9IENvbHVtbihTdHJpbmcoNTApLCBGb3JlaWduS2V5KCJkZXZpY2VzLmRldmljZV9pZCIpLCBudWxsYWJsZT1GYWxzZSwgaW5kZXg9VHJ1ZSkKICAgIHRpbWVzdGFtcCA9IENvbHVtbihEYXRlVGltZSwgZGVmYXVsdD1kYXRldGltZS51dGNub3csIGluZGV4PVRydWUpCiAgICBhbm9tYWx5X2ZsYWcgPSBDb2x1bW4oQm9vbGVhbiwgZGVmYXVsdD1GYWxzZSkKICAgIHNldmVyaXR5ID0gQ29sdW1uKFN0cmluZygyMCksIG51bGxhYmxlPUZhbHNlKSAgIyBsb3csIG1lZGl1bSwgaGlnaCwgY3JpdGljYWwKICAgIGFsZXJ0X3R5cGUgPSBDb2x1bW4oU3RyaW5nKDUwKSwgbnVsbGFibGU9RmFsc2UpICAjIHRlbXBfYW5vbWFseSwgdmlicmF0aW9uX3NwaWtlLCBiYXR0ZXJ5X2xvdywgZXRjLgogICAgZGVzY3JpcHRpb24gPSBDb2x1bW4oVGV4dCkKICAgIGFnZW50X3NvdXJjZSA9IENvbHVtbihTdHJpbmcoNTApKSAgIyB3aGljaCBhZ2VudCByYWlzZWQgaXQKICAgIHJlc29sdmVkID0gQ29sdW1uKEJvb2xlYW4sIGRlZmF1bHQ9RmFsc2UpCgogICAgZGV2aWNlID0gcmVsYXRpb25zaGlwKCJEZXZpY2UiLCBiYWNrX3BvcHVsYXRlcz0iYWxlcnRzIikKCgpjbGFzcyBJbmNpZGVudChCYXNlKToKICAgICIiIkNsYXNzaWZpZWQgc2VjdXJpdHkvb3BlcmF0aW9uYWwgaW5jaWRlbnRzIHdpdGggcmVzcG9uc2UgcmVjb21tZW5kYXRpb25zLiIiIgogICAgX190YWJsZW5hbWVfXyA9ICJpbmNpZGVudHMiCgogICAgaWQgPSBDb2x1bW4oSW50ZWdlciwgcHJpbWFyeV9rZXk9VHJ1ZSwgaW5kZXg9VHJ1ZSkKICAgIGRldmljZV9pZCA9IENvbHVtbihTdHJpbmcoNTApLCBGb3JlaWduS2V5KCJkZXZpY2VzLmRldmljZV9pZCIpLCBudWxsYWJsZT1GYWxzZSwgaW5kZXg9VHJ1ZSkKICAgIGRldGVjdGVkX2F0ID0gQ29sdW1uKERhdGVUaW1lLCBkZWZhdWx0PWRhdGV0aW1lLnV0Y25vdywgaW5kZXg9VHJ1ZSkKICAgIHRocmVhdF90eXBlID0gQ29sdW1uKFN0cmluZyg1MCksIG51bGxhYmxlPUZhbHNlKSAgIyB0YW1wZXJpbmcsIHVuYXV0aG9yaXplZF9maXJtd2FyZSwgYW5vbWFseV9jbHVzdGVyCiAgICBjbGFzc2lmaWNhdGlvbiA9IENvbHVtbihTdHJpbmcoMjApLCBudWxsYWJsZT1GYWxzZSkgICMgc2VjdXJpdHksIG9wZXJhdGlvbmFsLCBlbnZpcm9ubWVudGFsCiAgICBzZXZlcml0eSA9IENvbHVtbihTdHJpbmcoMjApLCBudWxsYWJsZT1GYWxzZSkKICAgIHN0YXR1cyA9IENvbHVtbihTdHJpbmcoMjApLCBkZWZhdWx0PSJvcGVuIikgICMgb3BlbiwgaW52ZXN0aWdhdGluZywgcmVzb2x2ZWQsIGZhbHNlX3Bvc2l0aXZlCiAgICByZWNvbW1lbmRhdGlvbiA9IENvbHVtbihUZXh0KSAgIyBSZXNwb25zZSBhY3Rpb24gcmVjb21tZW5kZWQgYnkgYWdlbnQKICAgIHJlc29sdmVkX2F0ID0gQ29sdW1uKERhdGVUaW1lLCBudWxsYWJsZT1UcnVlKQoKICAgIGRldmljZSA9IHJlbGF0aW9uc2hpcCgiRGV2aWNlIiwgYmFja19wb3B1bGF0ZXM9ImluY2lkZW50cyIpCgoKY2xhc3MgQWdlbnRUYXNrTG9nKEJhc2UpOgogICAgIiIiSWRlbnRpdHktYXdhcmUgYWdlbnQgYXVkaXQgdHJhaWwgLSBldmVyeSBkZWNpc2lvbiBpcyBsb2dnZWQgJiB2YWxpZGF0ZWQuIiIiCiAgICBfX3RhYmxlbmFtZV9fID0gImFnZW50X3Rhc2tfbG9ncyIKCiAgICBpZCA9IENvbHVtbihJbnRlZ2VyLCBwcmltYXJ5X2tleT1UcnVlLCBpbmRleD1UcnVlKQogICAgYWdlbnRfbmFtZSA9IENvbHVtbihTdHJpbmcoNTApLCBudWxsYWJsZT1GYWxzZSwgaW5kZXg9VHJ1ZSkKICAgIGFnZW50X3JvbGUgPSBDb2x1bW4oU3RyaW5nKDEwMCksIG51bGxhYmxlPUZhbHNlKQogICAgdGFzayA9IENvbHVtbihTdHJpbmcoMTAwKSwgbnVsbGFibGU9RmFsc2UpCiAgICBpbnB1dF9zdW1tYXJ5ID0gQ29sdW1uKFRleHQpCiAgICBvdXRwdXQgPSBDb2x1bW4oVGV4dCkKICAgIHRpbWVzdGFtcCA9IENvbHVtbihEYXRlVGltZSwgZGVmYXVsdD1kYXRldGltZS51dGNub3csIGluZGV4PVRydWUpCiAgICB2YWxpZGF0aW9uX3N0YXR1cyA9IENvbHVtbihTdHJpbmcoMjApLCBkZWZhdWx0PSJwZW5kaW5nIikgICMgcGVuZGluZywgdmFsaWRhdGVkLCByZWplY3RlZAogICAgdmFsaWRhdG9yX2FnZW50ID0gQ29sdW1uKFN0cmluZyg1MCksIG51bGxhYmxlPVRydWUpCiAgICBpZGVudGl0eV90b2tlbl9oYXNoID0gQ29sdW1uKFN0cmluZyg2NCkpICAjIGhhc2ggb2Ygc2lnbmluZyB0b2tlbiAobmV2ZXIgc3RvcmUgcmF3KQogICAgZXhlY3V0aW9uX3RpbWVfbXMgPSBDb2x1bW4oSW50ZWdlciwgZGVmYXVsdD0wKQoKCiMgLS0tIERCIHV0aWxpdGllcyAtLS0KCmRlZiBpbml0X2RiKCk6CiAgICAiIiJDcmVhdGUgYWxsIHRhYmxlcy4iIiIKICAgIEJhc2UubWV0YWRhdGEuY3JlYXRlX2FsbChiaW5kPWVuZ2luZSkKCgpkZWYgZ2V0X2RiKCk6CiAgICAiIiJGYXN0QVBJIGRlcGVuZGVuY3kgZm9yIERCIHNlc3Npb24uIiIiCiAgICBkYiA9IFNlc3Npb25Mb2NhbCgpCiAgICB0cnk6CiAgICAgICAgeWllbGQgZGIKICAgIGZpbmFsbHk6CiAgICAgICAgZGIuY2xvc2UoKQoKCmRlZiByZXNldF9kYigpOgogICAgIiIiRHJvcCBhbmQgcmVjcmVhdGUgYWxsIHRhYmxlcyAtIGZvciB0ZXN0cy9kZW1vcy4iIiIKICAgIEJhc2UubWV0YWRhdGEuZHJvcF9hbGwoYmluZD1lbmdpbmUpCiAgICBCYXNlLm1ldGFkYXRhLmNyZWF0ZV9hbGwoYmluZD1lbmdpbmUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHByaW50KCJJbml0aWFsaXppbmcgZGF0YWJhc2UuLi4iKQogICAgaW5pdF9kYigpCiAgICBwcmludChmIkRhdGFiYXNlIHJlYWR5IGF0OiB7REFUQUJBU0VfVVJMfSIpCg==", "backend/simulator/sensor_sim.py": "IiIiCklvVCBTZW5zb3IgU2ltdWxhdG9yCj09PT09PT09PT09PT09PT09PT09PQpTaW11bGF0ZXMgYSB3YXJlaG91c2UgZW52aXJvbm1lbnQgd2l0aCBtdWx0aXBsZSBJb1QgZGV2aWNlcyBnZW5lcmF0aW5nCnRlbXBlcmF0dXJlLCBodW1pZGl0eSwgdmlicmF0aW9uLCBiYXR0ZXJ5LCBhbmQgc2lnbmFsLXN0cmVuZ3RoIHJlYWRpbmdzLgoKSW5jbHVkZXMgZGVsaWJlcmF0ZSBhbm9tYWx5IGluamVjdGlvbiAofjglIGJ5IGRlZmF1bHQpIGZvciBNTCB0cmFpbmluZy9kZW1vLgoiIiIKCmltcG9ydCByYW5kb20KaW1wb3J0IG1hdGgKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWVkZWx0YQpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgRGljdAppbXBvcnQgb3MKCgojIFJlYWxpc3RpYyB3YXJlaG91c2Ugem9uZXMKV0FSRUhPVVNFX1pPTkVTID0gWwogICAgIldhcmVob3VzZS1BLVpvbmUtMSIsICJXYXJlaG91c2UtQS1ab25lLTIiLCAiV2FyZWhvdXNlLUEtWm9uZS0zIiwKICAgICJXYXJlaG91c2UtQi1Db2xkLVN0b3JhZ2UiLCAiV2FyZWhvdXNlLUItUmVjZWl2aW5nIiwKICAgICJXYXJlaG91c2UtQy1Mb2FkaW5nLUJheSIsICJXYXJlaG91c2UtQy1PZmZpY2UiLApdCgpERVZJQ0VfVFlQRVMgPSBbInRlbXBfc2Vuc29yIiwgImh1bWlkaXR5X3NlbnNvciIsICJ2aWJyYXRpb25fc2Vuc29yIiwgIm11bHRpX3NlbnNvciJdCkZJUk1XQVJFX1ZFUlNJT05TID0gWyIxLjIuMyIsICIxLjIuNCIsICIxLjMuMCIsICIxLjMuMSIsICIyLjAuMCJdCgoKZGVmIGdlbmVyYXRlX2RldmljZXMoY291bnQ6IGludCA9IDEyKSAtPiBMaXN0W0RpY3RdOgogICAgIiIiR2VuZXJhdGUgYSBmaXhlZCByb3N0ZXIgb2YgSW9UIGRldmljZXMuIiIiCiAgICBkZXZpY2VzID0gW10KICAgIGZvciBpIGluIHJhbmdlKGNvdW50KToKICAgICAgICBkZXZpY2UgPSB7CiAgICAgICAgICAgICJkZXZpY2VfaWQiOiBmIklPVC17MTAwMCArIGk6MDRkfSIsCiAgICAgICAgICAgICJuYW1lIjogZiJTZW5zb3Ite2Nocig2NSArIChpICUgMjYpKX17aTowMmR9IiwKICAgICAgICAgICAgImxvY2F0aW9uIjogcmFuZG9tLmNob2ljZShXQVJFSE9VU0VfWk9ORVMpLAogICAgICAgICAgICAiZGV2aWNlX3R5cGUiOiByYW5kb20uY2hvaWNlKERFVklDRV9UWVBFUyksCiAgICAgICAgICAgICJmaXJtd2FyZV92ZXJzaW9uIjogcmFuZG9tLmNob2ljZShGSVJNV0FSRV9WRVJTSU9OUyksCiAgICAgICAgICAgICJhdXRoX3N0YXR1cyI6ICJhdXRoZW50aWNhdGVkIiwKICAgICAgICAgICAgImJhdHRlcnlfbGV2ZWwiOiByb3VuZChyYW5kb20udW5pZm9ybSg0NS4wLCAxMDAuMCksIDEpLAogICAgICAgIH0KICAgICAgICBkZXZpY2VzLmFwcGVuZChkZXZpY2UpCiAgICByZXR1cm4gZGV2aWNlcwoKCmRlZiBnZW5lcmF0ZV9yZWFkaW5nKAogICAgZGV2aWNlX2lkOiBzdHIsCiAgICBsb2NhdGlvbjogc3RyLAogICAgaW5qZWN0X2Fub21hbHk6IGJvb2wgPSBGYWxzZSwKICAgIHRpbWVzdGFtcDogZGF0ZXRpbWUgPSBOb25lCikgLT4gRGljdDoKICAgICIiIgogICAgR2VuZXJhdGUgYSBzaW5nbGUgc2Vuc29yIHJlYWRpbmcuCgogICAgTm9ybWFsIHJhbmdlcyAod2FyZWhvdXNlKToKICAgICAgLSB0ZW1wZXJhdHVyZTogMTgtMjQgQyAoY29sZCBzdG9yYWdlOiAyLTggQykKICAgICAgLSBodW1pZGl0eTogNDAtNjAgJQogICAgICAtIHZpYnJhdGlvbjogMC4xLTIuMCBtbS9zCiAgICAgIC0gYmF0dGVyeTogZHJhaW5zIHNsb3dseQogICAgICAtIHNpZ25hbDogLTUwIHRvIC03NSBkQm0gKGdvb2QpCgogICAgQW5vbWFsaWVzOgogICAgICAtIHRlbXBlcmF0dXJlIHNwaWtlICg+MzUgb3IgPDApCiAgICAgIC0gaHVtaWRpdHkgZmxvb2QgKD44NSBvciA8MTUpCiAgICAgIC0gdmlicmF0aW9uIHNwaWtlICh0YW1wZXJpbmcsIGVxdWlwbWVudCBmYWlsdXJlKQogICAgICAtIHNpZ25hbCBkcm9wICgtOTArIGRCbSkKICAgICIiIgogICAgaWYgdGltZXN0YW1wIGlzIE5vbmU6CiAgICAgICAgdGltZXN0YW1wID0gZGF0ZXRpbWUudXRjbm93KCkKCiAgICBpc19jb2xkX3N0b3JhZ2UgPSAiQ29sZC1TdG9yYWdlIiBpbiBsb2NhdGlvbgoKICAgICMgQmFzZWxpbmUgcmVhZGluZ3MKICAgIHRlbXBfYmFzZWxpbmUgPSByYW5kb20udW5pZm9ybSgyLCA4KSBpZiBpc19jb2xkX3N0b3JhZ2UgZWxzZSByYW5kb20udW5pZm9ybSgxOCwgMjQpCiAgICBodW1pZGl0eV9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKDQwLCA2MCkKICAgIHZpYnJhdGlvbl9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKDAuMSwgMi4wKQogICAgc2lnbmFsX2Jhc2VsaW5lID0gcmFuZG9tLnVuaWZvcm0oLTc1LCAtNTApCiAgICBiYXR0ZXJ5X2Jhc2VsaW5lID0gcmFuZG9tLnVuaWZvcm0oNDAsIDEwMCkKCiAgICBpZiBpbmplY3RfYW5vbWFseToKICAgICAgICBhbm9tYWx5X3R5cGUgPSByYW5kb20uY2hvaWNlKFsKICAgICAgICAgICAgInRlbXBfc3Bpa2UiLCAidGVtcF9kcm9wIiwgImh1bWlkaXR5X2Zsb29kIiwgInZpYnJhdGlvbl9zcGlrZSIsCiAgICAgICAgICAgICJzaWduYWxfZHJvcCIsICJiYXR0ZXJ5X2NyaXRpY2FsIgogICAgICAgIF0pCiAgICAgICAgaWYgYW5vbWFseV90eXBlID09ICJ0ZW1wX3NwaWtlIjoKICAgICAgICAgICAgdGVtcF9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKDM1LCA1MCkKICAgICAgICBlbGlmIGFub21hbHlfdHlwZSA9PSAidGVtcF9kcm9wIjoKICAgICAgICAgICAgdGVtcF9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKC0xMCwgMCkKICAgICAgICBlbGlmIGFub21hbHlfdHlwZSA9PSAiaHVtaWRpdHlfZmxvb2QiOgogICAgICAgICAgICBodW1pZGl0eV9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKDg1LCA5OSkKICAgICAgICBlbGlmIGFub21hbHlfdHlwZSA9PSAidmlicmF0aW9uX3NwaWtlIjoKICAgICAgICAgICAgdmlicmF0aW9uX2Jhc2VsaW5lID0gcmFuZG9tLnVuaWZvcm0oOCwgMjApCiAgICAgICAgZWxpZiBhbm9tYWx5X3R5cGUgPT0gInNpZ25hbF9kcm9wIjoKICAgICAgICAgICAgc2lnbmFsX2Jhc2VsaW5lID0gcmFuZG9tLnVuaWZvcm0oLTEwMCwgLTkwKQogICAgICAgIGVsaWYgYW5vbWFseV90eXBlID09ICJiYXR0ZXJ5X2NyaXRpY2FsIjoKICAgICAgICAgICAgYmF0dGVyeV9iYXNlbGluZSA9IHJhbmRvbS51bmlmb3JtKDEsIDEwKQoKICAgICMgQWRkIHNtYWxsIG5vaXNlCiAgICByZXR1cm4gewogICAgICAgICJkZXZpY2VfaWQiOiBkZXZpY2VfaWQsCiAgICAgICAgInRpbWVzdGFtcCI6IHRpbWVzdGFtcC5pc29mb3JtYXQoKSwKICAgICAgICAidGVtcGVyYXR1cmUiOiByb3VuZCh0ZW1wX2Jhc2VsaW5lICsgcmFuZG9tLmdhdXNzKDAsIDAuMyksIDIpLAogICAgICAgICJodW1pZGl0eSI6IHJvdW5kKGh1bWlkaXR5X2Jhc2VsaW5lICsgcmFuZG9tLmdhdXNzKDAsIDEuMCksIDIpLAogICAgICAgICJ2aWJyYXRpb24iOiByb3VuZChtYXgoMCwgdmlicmF0aW9uX2Jhc2VsaW5lICsgcmFuZG9tLmdhdXNzKDAsIDAuMSkpLCAzKSwKICAgICAgICAiYmF0dGVyeSI6IHJvdW5kKG1heCgwLCBtaW4oMTAwLCBiYXR0ZXJ5X2Jhc2VsaW5lKSksIDEpLAogICAgICAgICJzaWduYWxfc3RyZW5ndGgiOiByb3VuZChzaWduYWxfYmFzZWxpbmUgKyByYW5kb20uZ2F1c3MoMCwgMiksIDEpLAogICAgfQoKCmRlZiBnZW5lcmF0ZV9iYXRjaCgKICAgIGRldmljZXM6IExpc3RbRGljdF0sCiAgICByZWFkaW5nc19wZXJfZGV2aWNlOiBpbnQgPSA1MCwKICAgIGFub21hbHlfcmF0ZTogZmxvYXQgPSAwLjA4LAogICAgdGltZV93aW5kb3dfaG91cnM6IGludCA9IDI0LAopIC0+IExpc3RbRGljdF06CiAgICAiIiIKICAgIEdlbmVyYXRlIGhpc3RvcmljYWwgYmF0Y2ggb2YgcmVhZGluZ3MgYWNyb3NzIGFsbCBkZXZpY2VzIG92ZXIgYSB0aW1lIHdpbmRvdy4KICAgIFVzZWZ1bCBmb3Igc2VlZGluZyBkZW1vIGRhdGEuCiAgICAiIiIKICAgIHJlYWRpbmdzID0gW10KICAgIGVuZF90aW1lID0gZGF0ZXRpbWUudXRjbm93KCkKICAgIHN0YXJ0X3RpbWUgPSBlbmRfdGltZSAtIHRpbWVkZWx0YShob3Vycz10aW1lX3dpbmRvd19ob3VycykKICAgIGludGVydmFsX3NlY29uZHMgPSAodGltZV93aW5kb3dfaG91cnMgKiAzNjAwKSAvIG1heChyZWFkaW5nc19wZXJfZGV2aWNlLCAxKQoKICAgIGZvciBkZXZpY2UgaW4gZGV2aWNlczoKICAgICAgICBmb3IgaSBpbiByYW5nZShyZWFkaW5nc19wZXJfZGV2aWNlKToKICAgICAgICAgICAgdHMgPSBzdGFydF90aW1lICsgdGltZWRlbHRhKHNlY29uZHM9aSAqIGludGVydmFsX3NlY29uZHMpCiAgICAgICAgICAgIGluamVjdCA9IHJhbmRvbS5yYW5kb20oKSA8IGFub21hbHlfcmF0ZQogICAgICAgICAgICByZWFkaW5nID0gZ2VuZXJhdGVfcmVhZGluZygKICAgICAgICAgICAgICAgIGRldmljZV9pZD1kZXZpY2VbImRldmljZV9pZCJdLAogICAgICAgICAgICAgICAgbG9jYXRpb249ZGV2aWNlWyJsb2NhdGlvbiJdLAogICAgICAgICAgICAgICAgaW5qZWN0X2Fub21hbHk9aW5qZWN0LAogICAgICAgICAgICAgICAgdGltZXN0YW1wPXRzLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlYWRpbmdzLmFwcGVuZChyZWFkaW5nKQoKICAgIHJldHVybiByZWFkaW5ncwoKCmRlZiBzaW11bGF0ZV9zZWN1cml0eV9ldmVudChkZXZpY2U6IERpY3QpIC0+IERpY3Q6CiAgICAiIiJHZW5lcmF0ZSBhIHNlY3VyaXR5LXJlbGV2YW50IGV2ZW50ICh0YW1wZXJpbmcsIHVuYXV0aG9yaXplZCBmaXJtd2FyZSwgYXV0aCBmYWlsKS4iIiIKICAgIGV2ZW50cyA9IFsKICAgICAgICB7CiAgICAgICAgICAgICJ0eXBlIjogInVuYXV0aG9yaXplZF9maXJtd2FyZSIsCiAgICAgICAgICAgICJkZXRhaWxzIjogZiJEZXZpY2Uge2RldmljZVsnZGV2aWNlX2lkJ119IHJlcG9ydGluZyBmaXJtd2FyZSB2ZXJzaW9uIDAuMC4xICh1bnNpZ25lZCkiLAogICAgICAgICAgICAiYXV0aF9zdGF0dXMiOiAic3VzcGljaW91cyIsCiAgICAgICAgfSwKICAgICAgICB7CiAgICAgICAgICAgICJ0eXBlIjogImF1dGhfZmFpbHVyZSIsCiAgICAgICAgICAgICJkZXRhaWxzIjogZiJEZXZpY2Uge2RldmljZVsnZGV2aWNlX2lkJ119IGZhaWxlZCBtdXR1YWwgVExTIGhhbmRzaGFrZSA1IHRpbWVzIiwKICAgICAgICAgICAgImF1dGhfc3RhdHVzIjogInVuYXV0aG9yaXplZCIsCiAgICAgICAgfSwKICAgICAgICB7CiAgICAgICAgICAgICJ0eXBlIjogInRhbXBlcmluZ19zdXNwZWN0ZWQiLAogICAgICAgICAgICAiZGV0YWlscyI6IGYiRGV2aWNlIHtkZXZpY2VbJ2RldmljZV9pZCddfSBwaHlzaWNhbCBzZWFsIHNlbnNvciB0cmlnZ2VyZWQiLAogICAgICAgICAgICAiYXV0aF9zdGF0dXMiOiAic3VzcGljaW91cyIsCiAgICAgICAgfSwKICAgIF0KICAgIHJldHVybiByYW5kb20uY2hvaWNlKGV2ZW50cykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgIyBRdWljayBzbW9rZSB0ZXN0CiAgICBkZXZpY2VzID0gZ2VuZXJhdGVfZGV2aWNlcyg1KQogICAgcHJpbnQoZiJHZW5lcmF0ZWQge2xlbihkZXZpY2VzKX0gZGV2aWNlcyIpCiAgICBwcmludCgiU2FtcGxlIGRldmljZToiLCBkZXZpY2VzWzBdKQoKICAgIHJlYWRpbmdzID0gZ2VuZXJhdGVfYmF0Y2goZGV2aWNlcywgcmVhZGluZ3NfcGVyX2RldmljZT0xMCwgYW5vbWFseV9yYXRlPTAuMikKICAgIHByaW50KGYiXG5HZW5lcmF0ZWQge2xlbihyZWFkaW5ncyl9IHJlYWRpbmdzIikKICAgIHByaW50KCJTYW1wbGUgcmVhZGluZzoiLCByZWFkaW5nc1swXSkK", "backend/ml/anomaly_detector.py": "IiIiCkFub21hbHkgRGV0ZWN0aW9uIC0gSXNvbGF0aW9uRm9yZXN0Cj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpMaWdodHdlaWdodCwgZGVmZW5zaWJsZSBhbm9tYWx5IGRldGVjdG9yIGZvciBJb1QgdGVsZW1ldHJ5LgpXaHkgSXNvbGF0aW9uRm9yZXN0OgogIC0gVW5zdXBlcnZpc2VkIChubyBsYWJlbGxlZCBhbm9tYWxpZXMgbmVlZGVkKQogIC0gRmFzdCBvbiB0YWJ1bGFyIGRhdGEKICAtIEhhbmRsZXMgbXVsdGl2YXJpYXRlIHdlbGwgKHRlbXAgKyBodW1pZGl0eSArIHZpYnJhdGlvbiBqb2ludGx5KQogIC0gSW5kdXN0cnkgc3RhbmRhcmQgZm9yIHRoaXMgZXhhY3QgdXNlIGNhc2UKIiIiCgppbXBvcnQgb3MKaW1wb3J0IGpvYmxpYgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4uZW5zZW1ibGUgaW1wb3J0IElzb2xhdGlvbkZvcmVzdApmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXIKZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIERpY3QsIFR1cGxlCgpNT0RFTF9ESVIgPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSwgInNhdmVkX21vZGVscyIpCm9zLm1ha2VkaXJzKE1PREVMX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKTU9ERUxfUEFUSCA9IG9zLnBhdGguam9pbihNT0RFTF9ESVIsICJpc29fZm9yZXN0LmpvYmxpYiIpClNDQUxFUl9QQVRIID0gb3MucGF0aC5qb2luKE1PREVMX0RJUiwgInNjYWxlci5qb2JsaWIiKQoKRkVBVFVSRVMgPSBbInRlbXBlcmF0dXJlIiwgImh1bWlkaXR5IiwgInZpYnJhdGlvbiIsICJiYXR0ZXJ5IiwgInNpZ25hbF9zdHJlbmd0aCJdCgoKY2xhc3MgQW5vbWFseURldGVjdG9yOgogICAgIiIiSXNvbGF0aW9uRm9yZXN0IHdyYXBwZXIgd2l0aCBzY2FsaW5nICsgcGVyc2lzdGVuY2UuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbnRhbWluYXRpb246IGZsb2F0ID0gMC4wOCwgcmFuZG9tX3N0YXRlOiBpbnQgPSA0Mik6CiAgICAgICAgc2VsZi5jb250YW1pbmF0aW9uID0gY29udGFtaW5hdGlvbgogICAgICAgIHNlbGYucmFuZG9tX3N0YXRlID0gcmFuZG9tX3N0YXRlCiAgICAgICAgc2VsZi5tb2RlbDogSXNvbGF0aW9uRm9yZXN0IHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLnNjYWxlcjogU3RhbmRhcmRTY2FsZXIgfCBOb25lID0gTm9uZQoKICAgIGRlZiBmaXQoc2VsZiwgcmVhZGluZ3M6IExpc3RbRGljdF0pIC0+IERpY3Q6CiAgICAgICAgIiIiVHJhaW4gb24gYSBsaXN0IG9mIHJlYWRpbmcgZGljdHMuIiIiCiAgICAgICAgaWYgbGVuKHJlYWRpbmdzKSA8IDIwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTmVlZCBhdCBsZWFzdCAyMCByZWFkaW5ncyB0byB0cmFpbiwgZ290IHtsZW4ocmVhZGluZ3MpfSIpCgogICAgICAgIGRmID0gcGQuRGF0YUZyYW1lKHJlYWRpbmdzKQogICAgICAgIG1pc3NpbmcgPSBbZiBmb3IgZiBpbiBGRUFUVVJFUyBpZiBmIG5vdCBpbiBkZi5jb2x1bW5zXQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNaXNzaW5nIGZlYXR1cmVzIGluIHRyYWluaW5nIGRhdGE6IHttaXNzaW5nfSIpCgogICAgICAgIFggPSBkZltGRUFUVVJFU10udmFsdWVzCiAgICAgICAgc2VsZi5zY2FsZXIgPSBTdGFuZGFyZFNjYWxlcigpCiAgICAgICAgWF9zY2FsZWQgPSBzZWxmLnNjYWxlci5maXRfdHJhbnNmb3JtKFgpCgogICAgICAgIHNlbGYubW9kZWwgPSBJc29sYXRpb25Gb3Jlc3QoCiAgICAgICAgICAgIGNvbnRhbWluYXRpb249c2VsZi5jb250YW1pbmF0aW9uLAogICAgICAgICAgICByYW5kb21fc3RhdGU9c2VsZi5yYW5kb21fc3RhdGUsCiAgICAgICAgICAgIG5fZXN0aW1hdG9ycz0xMDAsCiAgICAgICAgKQogICAgICAgIHNlbGYubW9kZWwuZml0KFhfc2NhbGVkKQoKICAgICAgICAjIFNlbGYtZXZhbHVhdGlvbiBvbiB0cmFpbmluZyBzZXQKICAgICAgICBwcmVkcyA9IHNlbGYubW9kZWwucHJlZGljdChYX3NjYWxlZCkKICAgICAgICBhbm9tYWxpZXMgPSAocHJlZHMgPT0gLTEpLnN1bSgpCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJ0cmFpbmVkIjogVHJ1ZSwKICAgICAgICAgICAgIm5fc2FtcGxlcyI6IGxlbihyZWFkaW5ncyksCiAgICAgICAgICAgICJuX2ZlYXR1cmVzIjogbGVuKEZFQVRVUkVTKSwKICAgICAgICAgICAgImFub21hbGllc19kZXRlY3RlZCI6IGludChhbm9tYWxpZXMpLAogICAgICAgICAgICAiYW5vbWFseV9yYXRlIjogZmxvYXQoYW5vbWFsaWVzIC8gbGVuKHJlYWRpbmdzKSksCiAgICAgICAgICAgICJjb250YW1pbmF0aW9uIjogc2VsZi5jb250YW1pbmF0aW9uLAogICAgICAgIH0KCiAgICBkZWYgcHJlZGljdChzZWxmLCByZWFkaW5nOiBEaWN0KSAtPiBUdXBsZVtib29sLCBmbG9hdF06CiAgICAgICAgIiIiCiAgICAgICAgUHJlZGljdCBpZiBhIHNpbmdsZSByZWFkaW5nIGlzIGFub21hbG91cy4KICAgICAgICBSZXR1cm5zOiAoaXNfYW5vbWFseTogYm9vbCwgYW5vbWFseV9zY29yZTogZmxvYXQgaW4gWy0xLCAxXSwgbG93ZXIgPSBtb3JlIGFub21hbG91cykKICAgICAgICAiIiIKICAgICAgICBpZiBzZWxmLm1vZGVsIGlzIE5vbmUgb3Igc2VsZi5zY2FsZXIgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJNb2RlbCBub3QgdHJhaW5lZC4gQ2FsbCBmaXQoKSBvciBsb2FkKCkgZmlyc3QuIikKCiAgICAgICAgWCA9IG5wLmFycmF5KFtbcmVhZGluZy5nZXQoZiwgMC4wKSBmb3IgZiBpbiBGRUFUVVJFU11dKQogICAgICAgIFhfc2NhbGVkID0gc2VsZi5zY2FsZXIudHJhbnNmb3JtKFgpCiAgICAgICAgcHJlZCA9IHNlbGYubW9kZWwucHJlZGljdChYX3NjYWxlZClbMF0KICAgICAgICBzY29yZSA9IHNlbGYubW9kZWwuc2NvcmVfc2FtcGxlcyhYX3NjYWxlZClbMF0KICAgICAgICByZXR1cm4gYm9vbChwcmVkID09IC0xKSwgZmxvYXQoc2NvcmUpCgogICAgZGVmIHByZWRpY3RfYmF0Y2goc2VsZiwgcmVhZGluZ3M6IExpc3RbRGljdF0pIC0+IExpc3RbVHVwbGVbYm9vbCwgZmxvYXRdXToKICAgICAgICAiIiJWZWN0b3Jpc2VkIHByZWRpY3Rpb24gZm9yIG1hbnkgcmVhZGluZ3MuIiIiCiAgICAgICAgaWYgc2VsZi5tb2RlbCBpcyBOb25lIG9yIHNlbGYuc2NhbGVyIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTW9kZWwgbm90IHRyYWluZWQuIikKICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZShyZWFkaW5ncykKICAgICAgICBYID0gZGZbRkVBVFVSRVNdLnZhbHVlcwogICAgICAgIFhfc2NhbGVkID0gc2VsZi5zY2FsZXIudHJhbnNmb3JtKFgpCiAgICAgICAgcHJlZHMgPSBzZWxmLm1vZGVsLnByZWRpY3QoWF9zY2FsZWQpCiAgICAgICAgc2NvcmVzID0gc2VsZi5tb2RlbC5zY29yZV9zYW1wbGVzKFhfc2NhbGVkKQogICAgICAgIHJldHVybiBbKHAgPT0gLTEsIGZsb2F0KHMpKSBmb3IgcCwgcyBpbiB6aXAocHJlZHMsIHNjb3JlcyldCgogICAgZGVmIHNhdmUoc2VsZik6CiAgICAgICAgIiIiUGVyc2lzdCBtb2RlbCArIHNjYWxlci4iIiIKICAgICAgICBpZiBzZWxmLm1vZGVsIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm90aGluZyB0byBzYXZlIC0gbW9kZWwgbm90IHRyYWluZWQiKQogICAgICAgIGpvYmxpYi5kdW1wKHNlbGYubW9kZWwsIE1PREVMX1BBVEgpCiAgICAgICAgam9ibGliLmR1bXAoc2VsZi5zY2FsZXIsIFNDQUxFUl9QQVRIKQogICAgICAgIHJldHVybiB7Im1vZGVsIjogTU9ERUxfUEFUSCwgInNjYWxlciI6IFNDQUxFUl9QQVRIfQoKICAgIGRlZiBsb2FkKHNlbGYpIC0+IGJvb2w6CiAgICAgICAgIiIiTG9hZCBwZXJzaXN0ZWQgbW9kZWwgKyBzY2FsZXIuIFJldHVybnMgVHJ1ZSBpZiBzdWNjZXNzZnVsLiIiIgogICAgICAgIGlmIG5vdCAob3MucGF0aC5leGlzdHMoTU9ERUxfUEFUSCkgYW5kIG9zLnBhdGguZXhpc3RzKFNDQUxFUl9QQVRIKSk6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHNlbGYubW9kZWwgPSBqb2JsaWIubG9hZChNT0RFTF9QQVRIKQogICAgICAgIHNlbGYuc2NhbGVyID0gam9ibGliLmxvYWQoU0NBTEVSX1BBVEgpCiAgICAgICAgcmV0dXJuIFRydWUKCgpkZWYgY2xhc3NpZnlfc2V2ZXJpdHkoc2NvcmU6IGZsb2F0KSAtPiBzdHI6CiAgICAiIiIKICAgIE1hcCBhbm9tYWx5IHNjb3JlIHRvIHNldmVyaXR5LiBMb3dlciBzY29yZXMgPSBtb3JlIGFub21hbG91cy4KICAgIENhbGlicmF0ZWQgYWdhaW5zdCBJc29sYXRpb25Gb3Jlc3Qgb3V0cHV0IHJhbmdlICh+LTAuOCB0byAtMC4zIHR5cGljYWwpLgogICAgIiIiCiAgICBpZiBzY29yZSA8IC0wLjY1OgogICAgICAgIHJldHVybiAiY3JpdGljYWwiCiAgICBlbGlmIHNjb3JlIDwgLTAuNTU6CiAgICAgICAgcmV0dXJuICJoaWdoIgogICAgZWxpZiBzY29yZSA8IC0wLjQ1OgogICAgICAgIHJldHVybiAibWVkaXVtIgogICAgZWxzZToKICAgICAgICByZXR1cm4gImxvdyIKCgpkZWYgY2xhc3NpZnlfYWxlcnRfdHlwZShyZWFkaW5nOiBEaWN0KSAtPiBzdHI6CiAgICAiIiJIZXVyaXN0aWNhbGx5IGNsYXNzaWZ5IHdoYXQga2luZCBvZiBhbm9tYWx5IGJhc2VkIG9uIHdoaWNoIGZlYXR1cmUgaXMgZXh0cmVtZS4iIiIKICAgIGlmIHJlYWRpbmcuZ2V0KCJ0ZW1wZXJhdHVyZSIsIDIwKSA+IDM1OgogICAgICAgIHJldHVybiAidGVtcF9zcGlrZSIKICAgIGlmIHJlYWRpbmcuZ2V0KCJ0ZW1wZXJhdHVyZSIsIDIwKSA8IDA6CiAgICAgICAgcmV0dXJuICJ0ZW1wX2Ryb3AiCiAgICBpZiByZWFkaW5nLmdldCgiaHVtaWRpdHkiLCA1MCkgPiA4NToKICAgICAgICByZXR1cm4gImh1bWlkaXR5X2Zsb29kIgogICAgaWYgcmVhZGluZy5nZXQoImh1bWlkaXR5IiwgNTApIDwgMTU6CiAgICAgICAgcmV0dXJuICJodW1pZGl0eV9kcm91Z2h0IgogICAgaWYgcmVhZGluZy5nZXQoInZpYnJhdGlvbiIsIDEpID4gODoKICAgICAgICByZXR1cm4gInZpYnJhdGlvbl9zcGlrZSIKICAgIGlmIHJlYWRpbmcuZ2V0KCJiYXR0ZXJ5IiwgNTApIDwgMTA6CiAgICAgICAgcmV0dXJuICJiYXR0ZXJ5X2NyaXRpY2FsIgogICAgaWYgcmVhZGluZy5nZXQoInNpZ25hbF9zdHJlbmd0aCIsIC02NSkgPCAtOTA6CiAgICAgICAgcmV0dXJuICJzaWduYWxfbG9zcyIKICAgIHJldHVybiAibXVsdGl2YXJpYXRlX2Fub21hbHkiCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgICMgU21va2UgdGVzdAogICAgaW1wb3J0IHN5cwogICAgc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pLCAiLi4iLCAiLi4iKSkKICAgIGZyb20gYmFja2VuZC5zaW11bGF0b3Iuc2Vuc29yX3NpbSBpbXBvcnQgZ2VuZXJhdGVfZGV2aWNlcywgZ2VuZXJhdGVfYmF0Y2gKCiAgICBwcmludCgiW01MXSBHZW5lcmF0aW5nIHRyYWluaW5nIGRhdGEuLi4iKQogICAgZGV2aWNlcyA9IGdlbmVyYXRlX2RldmljZXMoMTApCiAgICByZWFkaW5ncyA9IGdlbmVyYXRlX2JhdGNoKGRldmljZXMsIHJlYWRpbmdzX3Blcl9kZXZpY2U9NTAsIGFub21hbHlfcmF0ZT0wLjA4KQogICAgcHJpbnQoZiJbTUxdIEdvdCB7bGVuKHJlYWRpbmdzKX0gcmVhZGluZ3MiKQoKICAgIGRldGVjdG9yID0gQW5vbWFseURldGVjdG9yKGNvbnRhbWluYXRpb249MC4wOCkKICAgIHN0YXRzID0gZGV0ZWN0b3IuZml0KHJlYWRpbmdzKQogICAgcHJpbnQoZiJbTUxdIFRyYWluaW5nIHN0YXRzOiB7c3RhdHN9IikKCiAgICBkZXRlY3Rvci5zYXZlKCkKICAgIHByaW50KGYiW01MXSBTYXZlZCB0byB7TU9ERUxfUEFUSH0iKQoKICAgICMgVGVzdCBzaW5nbGUgcHJlZGljdGlvbgogICAgdGVzdF9ub3JtYWwgPSB7InRlbXBlcmF0dXJlIjogMjEuMCwgImh1bWlkaXR5IjogNTAuMCwgInZpYnJhdGlvbiI6IDEuMCwgImJhdHRlcnkiOiA4MC4wLCAic2lnbmFsX3N0cmVuZ3RoIjogLTY1LjB9CiAgICB0ZXN0X2Fub21hbHkgPSB7InRlbXBlcmF0dXJlIjogNDUuMCwgImh1bWlkaXR5IjogOTUuMCwgInZpYnJhdGlvbiI6IDE1LjAsICJiYXR0ZXJ5IjogNS4wLCAic2lnbmFsX3N0cmVuZ3RoIjogLTk1LjB9CgogICAgaXNfYW5vbSwgc2NvcmUgPSBkZXRlY3Rvci5wcmVkaWN0KHRlc3Rfbm9ybWFsKQogICAgcHJpbnQoZiJbTUxdIE5vcm1hbCByZWFkaW5nIOKGkiBhbm9tYWx5PXtpc19hbm9tfSwgc2NvcmU9e3Njb3JlOi4zZn0sIHNldmVyaXR5PXtjbGFzc2lmeV9zZXZlcml0eShzY29yZSl9IikKICAgIGlzX2Fub20sIHNjb3JlID0gZGV0ZWN0b3IucHJlZGljdCh0ZXN0X2Fub21hbHkpCiAgICBwcmludChmIltNTF0gRXh0cmVtZSByZWFkaW5nIOKGkiBhbm9tYWx5PXtpc19hbm9tfSwgc2NvcmU9e3Njb3JlOi4zZn0sIHNldmVyaXR5PXtjbGFzc2lmeV9zZXZlcml0eShzY29yZSl9IikK", "backend/auth/identity.py": "IiIiCklkZW50aXR5LUF3YXJlIEFnZW50IEF1dGhlbnRpY2F0aW9uCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpJbXBsZW1lbnRzIHRoZSAiSWRlbnRpdHkgQXdhcmUgQWdlbnQgRGV2ZWxvcG1lbnQgUmVxdWlyZW1lbnQiIGZyb20gdGhlIHNwZWMuCgpFYWNoIGFnZW50IGdldHM6CiAgLSBBIHNpZ25lZCBKV1Qgd2l0aCByb2xlICsgcGVybWlzc2lvbnMgY2xhaW1zCiAgLSBBIHNjb3BlZCBwZXJtaXNzaW9uIHNldCAocmVhZC1vbmx5LCB3cml0ZSwgYWRtaW4pCiAgLSBBbiBhdWRpdCB0cmFpbCBlbnRyeSBmb3IgZXZlcnkgZGVjaXNpb24KIiIiCgppbXBvcnQgb3MKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IHNlY3JldHMKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWVkZWx0YSwgdGltZXpvbmUKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIExpc3QsIE9wdGlvbmFsCmltcG9ydCBqd3QKaW1wb3J0IGJjcnlwdApmcm9tIGRvdGVudiBpbXBvcnQgbG9hZF9kb3RlbnYKCmxvYWRfZG90ZW52KCkKCkpXVF9TRUNSRVQgPSBvcy5nZXRlbnYoIkpXVF9TRUNSRVQiLCAiZGV2LXNlY3JldC1jaGFuZ2UtbWUiKQpKV1RfQUxHT1JJVEhNID0gb3MuZ2V0ZW52KCJKV1RfQUxHT1JJVEhNIiwgIkhTMjU2IikKSldUX0VYUElSWV9NSU5VVEVTID0gaW50KG9zLmdldGVudigiSldUX0VYUElSWV9NSU5VVEVTIiwgIjYwIikpCgpBR0VOVF9TSUdOSU5HX0tFWSA9IG9zLmdldGVudigiQUdFTlRfU0lHTklOR19LRVkiKSBvciBzZWNyZXRzLnRva2VuX3VybHNhZmUoMzIpCgoKIyAtLS0gQWdlbnQgaWRlbnRpdHkgcmVnaXN0cnkgLS0tCgpBR0VOVF9QRVJNSVNTSU9OUyA9IHsKICAgICJ0ZWxlbWV0cnlfaW5nZXN0aW9uIjogewogICAgICAgICJyb2xlIjogIlRlbGVtZXRyeSBJbmdlc3Rpb24gU3BlY2lhbGlzdCIsCiAgICAgICAgInBlcm1pc3Npb25zIjogWyJyZWFkOmRldmljZXMiLCAicmVhZDpzZW5zb3JfcmVhZGluZ3MiLCAid3JpdGU6c2Vuc29yX3JlYWRpbmdzIl0sCiAgICB9LAogICAgImRldmljZV9oZWFsdGgiOiB7CiAgICAgICAgInJvbGUiOiAiRGV2aWNlIEhlYWx0aCBNb25pdG9yIiwKICAgICAgICAicGVybWlzc2lvbnMiOiBbInJlYWQ6ZGV2aWNlcyIsICJyZWFkOnNlbnNvcl9yZWFkaW5ncyIsICJ3cml0ZTphbGVydHMiXSwKICAgIH0sCiAgICAiYW5vbWFseV9kZXRlY3RvciI6IHsKICAgICAgICAicm9sZSI6ICJBbm9tYWx5IERldGVjdGlvbiBTcGVjaWFsaXN0IiwKICAgICAgICAicGVybWlzc2lvbnMiOiBbInJlYWQ6c2Vuc29yX3JlYWRpbmdzIiwgIndyaXRlOmFsZXJ0cyJdLAogICAgfSwKICAgICJzZWN1cml0eSI6IHsKICAgICAgICAicm9sZSI6ICJTZWN1cml0eSBPcGVyYXRpb25zIEFnZW50IiwKICAgICAgICAicGVybWlzc2lvbnMiOiBbInJlYWQ6ZGV2aWNlcyIsICJyZWFkOnNlbnNvcl9yZWFkaW5ncyIsICJ3cml0ZTppbmNpZGVudHMiLCAid3JpdGU6YWxlcnRzIl0sCiAgICB9LAogICAgImluY2lkZW50X2NsYXNzaWZpZXIiOiB7CiAgICAgICAgInJvbGUiOiAiSW5jaWRlbnQgQ2xhc3NpZmljYXRpb24gU3BlY2lhbGlzdCIsCiAgICAgICAgInBlcm1pc3Npb25zIjogWyJyZWFkOmFsZXJ0cyIsICJ3cml0ZTppbmNpZGVudHMiXSwKICAgIH0sCiAgICAicmVzcG9uc2VfcmVjb21tZW5kZXIiOiB7CiAgICAgICAgInJvbGUiOiAiUmVzcG9uc2UgUmVjb21tZW5kYXRpb24gRW5naW5lIiwKICAgICAgICAicGVybWlzc2lvbnMiOiBbInJlYWQ6aW5jaWRlbnRzIiwgInJlYWQ6YWxlcnRzIiwgIndyaXRlOmluY2lkZW50cyJdLAogICAgfSwKICAgICJ2YWxpZGF0b3IiOiB7CiAgICAgICAgInJvbGUiOiAiRGVjaXNpb24gVmFsaWRhdG9yIC8gUmV2aWV3ZXIiLAogICAgICAgICJwZXJtaXNzaW9ucyI6IFsicmVhZDphZ2VudF90YXNrX2xvZ3MiLCAid3JpdGU6YWdlbnRfdGFza19sb2dzIl0sCiAgICB9LAp9CgoKZGVmIGlzc3VlX2FnZW50X3Rva2VuKGFnZW50X25hbWU6IHN0cikgLT4gc3RyOgogICAgIiIiSXNzdWUgYSBzaWduZWQgSldUIGlkZW50aXR5IHRva2VuIGZvciBhbiBhZ2VudC4iIiIKICAgIGlmIGFnZW50X25hbWUgbm90IGluIEFHRU5UX1BFUk1JU1NJT05TOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmtub3duIGFnZW50OiB7YWdlbnRfbmFtZX0uIEtub3duOiB7bGlzdChBR0VOVF9QRVJNSVNTSU9OUyl9IikKCiAgICBwcm9maWxlID0gQUdFTlRfUEVSTUlTU0lPTlNbYWdlbnRfbmFtZV0KICAgIHBheWxvYWQgPSB7CiAgICAgICAgInN1YiI6IGFnZW50X25hbWUsCiAgICAgICAgInJvbGUiOiBwcm9maWxlWyJyb2xlIl0sCiAgICAgICAgInBlcm1pc3Npb25zIjogcHJvZmlsZVsicGVybWlzc2lvbnMiXSwKICAgICAgICAiaWF0IjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YyksCiAgICAgICAgImV4cCI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpICsgdGltZWRlbHRhKG1pbnV0ZXM9SldUX0VYUElSWV9NSU5VVEVTKSwKICAgICAgICAidHlwZSI6ICJhZ2VudF9pZGVudGl0eSIsCiAgICB9CiAgICByZXR1cm4gand0LmVuY29kZShwYXlsb2FkLCBBR0VOVF9TSUdOSU5HX0tFWSwgYWxnb3JpdGhtPUpXVF9BTEdPUklUSE0pCgoKZGVmIHZlcmlmeV9hZ2VudF90b2tlbih0b2tlbjogc3RyKSAtPiBPcHRpb25hbFtEaWN0XToKICAgICIiIlZlcmlmeSBhbiBhZ2VudCB0b2tlbiBhbmQgcmV0dXJuIGl0cyBjbGFpbXMsIG9yIE5vbmUgaWYgaW52YWxpZC4iIiIKICAgIHRyeToKICAgICAgICByZXR1cm4gand0LmRlY29kZSh0b2tlbiwgQUdFTlRfU0lHTklOR19LRVksIGFsZ29yaXRobXM9W0pXVF9BTEdPUklUSE1dKQogICAgZXhjZXB0IGp3dC5QeUpXVEVycm9yOgogICAgICAgIHJldHVybiBOb25lCgoKZGVmIGhhc2hfdG9rZW4odG9rZW46IHN0cikgLT4gc3RyOgogICAgIiIiU0hBLTI1NiBoYXNoIG9mIHRva2VuIGZvciBhdWRpdCBsb2dnaW5nIChuZXZlciBzdG9yZSByYXcgdG9rZW5zKS4iIiIKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nih0b2tlbi5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgaGFzX3Blcm1pc3Npb24odG9rZW46IHN0ciwgcmVxdWlyZWRfcGVybWlzc2lvbjogc3RyKSAtPiBib29sOgogICAgIiIiQ2hlY2sgaWYgYW4gYWdlbnQgdG9rZW4gZ3JhbnRzIGEgc3BlY2lmaWMgcGVybWlzc2lvbi4iIiIKICAgIGNsYWltcyA9IHZlcmlmeV9hZ2VudF90b2tlbih0b2tlbikKICAgIGlmIG5vdCBjbGFpbXM6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gcmVxdWlyZWRfcGVybWlzc2lvbiBpbiBjbGFpbXMuZ2V0KCJwZXJtaXNzaW9ucyIsIFtdKQoKCiMgLS0tIFVzZXIgYXV0aCAoZm9yIGRhc2hib2FyZCBhZG1pbiBwYWdlKSAtLS0KCmRlZiBoYXNoX3Bhc3N3b3JkKHBhc3N3b3JkOiBzdHIpIC0+IHN0cjoKICAgICMgVHJ1bmNhdGUgdG8gYmNyeXB0J3MgNzItYnl0ZSBtYXggZGVmZW5zaXZlbHkKICAgIHB3X2J5dGVzID0gcGFzc3dvcmQuZW5jb2RlKCJ1dGYtOCIpWzo3Ml0KICAgIHJldHVybiBiY3J5cHQuaGFzaHB3KHB3X2J5dGVzLCBiY3J5cHQuZ2Vuc2FsdCgpKS5kZWNvZGUoInV0Zi04IikKCgpkZWYgdmVyaWZ5X3Bhc3N3b3JkKHBsYWluOiBzdHIsIGhhc2hlZDogc3RyKSAtPiBib29sOgogICAgcHdfYnl0ZXMgPSBwbGFpbi5lbmNvZGUoInV0Zi04IilbOjcyXQogICAgdHJ5OgogICAgICAgIHJldHVybiBiY3J5cHQuY2hlY2twdyhwd19ieXRlcywgaGFzaGVkLmVuY29kZSgidXRmLTgiKSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGlzc3VlX3VzZXJfdG9rZW4odXNlcm5hbWU6IHN0ciwgcm9sZTogc3RyID0gIm9wZXJhdG9yIikgLT4gc3RyOgogICAgIiIiSXNzdWUgYSBKV1QgZm9yIGEgaHVtYW4gdXNlciAoZGFzaGJvYXJkIGF1dGgpLiIiIgogICAgcGF5bG9hZCA9IHsKICAgICAgICAic3ViIjogdXNlcm5hbWUsCiAgICAgICAgInJvbGUiOiByb2xlLAogICAgICAgICJpYXQiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKSwKICAgICAgICAiZXhwIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykgKyB0aW1lZGVsdGEobWludXRlcz1KV1RfRVhQSVJZX01JTlVURVMpLAogICAgICAgICJ0eXBlIjogInVzZXIiLAogICAgfQogICAgcmV0dXJuIGp3dC5lbmNvZGUocGF5bG9hZCwgSldUX1NFQ1JFVCwgYWxnb3JpdGhtPUpXVF9BTEdPUklUSE0pCgoKZGVmIHZlcmlmeV91c2VyX3Rva2VuKHRva2VuOiBzdHIpIC0+IE9wdGlvbmFsW0RpY3RdOgogICAgdHJ5OgogICAgICAgIHJldHVybiBqd3QuZGVjb2RlKHRva2VuLCBKV1RfU0VDUkVULCBhbGdvcml0aG1zPVtKV1RfQUxHT1JJVEhNXSkKICAgIGV4Y2VwdCBqd3QuUHlKV1RFcnJvcjoKICAgICAgICByZXR1cm4gTm9uZQoKCiMgLS0tIERlZmF1bHQgYWRtaW4gKGZvciBkZW1vKSAtLS0KREVGQVVMVF9VU0VSUyA9IHsKICAgIG9zLmdldGVudigiQURNSU5fVVNFUk5BTUUiLCAiYWRtaW4iKTogewogICAgICAgICJwYXNzd29yZF9oYXNoIjogaGFzaF9wYXNzd29yZChvcy5nZXRlbnYoIkFETUlOX1BBU1NXT1JEIiwgImFkbWluMTIzIikpLAogICAgICAgICJyb2xlIjogImFkbWluIiwKICAgIH0sCiAgICAib3BlcmF0b3IiOiB7CiAgICAgICAgInBhc3N3b3JkX2hhc2giOiBoYXNoX3Bhc3N3b3JkKCJvcGVyYXRvcjEyMyIpLAogICAgICAgICJyb2xlIjogIm9wZXJhdG9yIiwKICAgIH0sCn0KCgpkZWYgYXV0aGVudGljYXRlX3VzZXIodXNlcm5hbWU6IHN0ciwgcGFzc3dvcmQ6IHN0cikgLT4gT3B0aW9uYWxbc3RyXToKICAgICIiIlJldHVybnMgSldUIHRva2VuIGlmIGNyZWRlbnRpYWxzIHZhbGlkLCBlbHNlIE5vbmUuIiIiCiAgICB1c2VyID0gREVGQVVMVF9VU0VSUy5nZXQodXNlcm5hbWUpCiAgICBpZiBub3QgdXNlcjoKICAgICAgICByZXR1cm4gTm9uZQogICAgaWYgbm90IHZlcmlmeV9wYXNzd29yZChwYXNzd29yZCwgdXNlclsicGFzc3dvcmRfaGFzaCJdKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGlzc3VlX3VzZXJfdG9rZW4odXNlcm5hbWUsIHVzZXJbInJvbGUiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgIyBTbW9rZSB0ZXN0CiAgICBwcmludCgiPT09IEFnZW50IElkZW50aXR5IFRlc3QgPT09IikKICAgIGZvciBhZ2VudCBpbiBBR0VOVF9QRVJNSVNTSU9OUzoKICAgICAgICB0b2tlbiA9IGlzc3VlX2FnZW50X3Rva2VuKGFnZW50KQogICAgICAgIGNsYWltcyA9IHZlcmlmeV9hZ2VudF90b2tlbih0b2tlbikKICAgICAgICBwcmludChmIiAge2FnZW50fTogcm9sZT17Y2xhaW1zWydyb2xlJ119LCBwZXJtcz17bGVuKGNsYWltc1sncGVybWlzc2lvbnMnXSl9IikKCiAgICBwcmludCgiXG49PT0gUGVybWlzc2lvbiBDaGVjayA9PT0iKQogICAgc2VjX3Rva2VuID0gaXNzdWVfYWdlbnRfdG9rZW4oInNlY3VyaXR5IikKICAgIHByaW50KGYiICBzZWN1cml0eSBjYW4gd3JpdGU6aW5jaWRlbnRzIOKGkiB7aGFzX3Blcm1pc3Npb24oc2VjX3Rva2VuLCAnd3JpdGU6aW5jaWRlbnRzJyl9IikKICAgIHByaW50KGYiICBzZWN1cml0eSBjYW4gd3JpdGU6ZGV2aWNlcyDihpIge2hhc19wZXJtaXNzaW9uKHNlY190b2tlbiwgJ3dyaXRlOmRldmljZXMnKX0iKQoKICAgIHByaW50KCJcbj09PSBVc2VyIEF1dGggVGVzdCA9PT0iKQogICAgdG9rID0gYXV0aGVudGljYXRlX3VzZXIoImFkbWluIiwgImFkbWluMTIzIikKICAgIHByaW50KGYiICBhZG1pbiBsb2dpbiDihpIge3Rva1s6NDBdfS4uLiIpCiAgICB0b2sgPSBhdXRoZW50aWNhdGVfdXNlcigiYWRtaW4iLCAid3JvbmciKQogICAgcHJpbnQoZiIgIGJhZCBsb2dpbiDihpIge3Rva30iKQo=", "backend/outputs.py": "IiIiCk91dHB1dCBHZW5lcmF0b3JzCj09PT09PT09PT09PT09PT09PQpUd28gc2ltcGxlLCB2aXN1YWwgb3V0cHV0czoKICAxLiBRUiBDb2RlICAtIHBlciBkZXZpY2UsIGxpbmtzIHRvIGl0cyBkYXNoYm9hcmQgcGFnZQogIDIuIFBERiBSZXBvcnQgLSBwcm9mZXNzaW9uYWwgaW5jaWRlbnQgcmVwb3J0IGFmdGVyIGVhY2ggY3JldyBydW4KIiIiCgppbXBvcnQgaW8KaW1wb3J0IG9zCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBEaWN0LCBPcHRpb25hbAoKaW1wb3J0IHFyY29kZQpmcm9tIHJlcG9ydGxhYi5saWIucGFnZXNpemVzIGltcG9ydCBBNApmcm9tIHJlcG9ydGxhYi5saWIgaW1wb3J0IGNvbG9ycwpmcm9tIHJlcG9ydGxhYi5saWIudW5pdHMgaW1wb3J0IG1tCmZyb20gcmVwb3J0bGFiLmxpYi5zdHlsZXMgaW1wb3J0IGdldFNhbXBsZVN0eWxlU2hlZXQsIFBhcmFncmFwaFN0eWxlCmZyb20gcmVwb3J0bGFiLnBsYXR5cHVzIGltcG9ydCAoCiAgICBTaW1wbGVEb2NUZW1wbGF0ZSwgUGFyYWdyYXBoLCBTcGFjZXIsIFRhYmxlLCBUYWJsZVN0eWxlLCBIUkZsb3dhYmxlCikKZnJvbSByZXBvcnRsYWIubGliLmVudW1zIGltcG9ydCBUQV9DRU5URVIsIFRBX0xFRlQKCgojIOKUgOKUgCBRUiBDb2RlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIGdlbmVyYXRlX2RldmljZV9xcihkZXZpY2VfaWQ6IHN0ciwgZGFzaGJvYXJkX2Jhc2VfdXJsOiBzdHIpIC0+IGJ5dGVzOgogICAgIiIiCiAgICBHZW5lcmF0ZSBhIFFSIGNvZGUgUE5HIChieXRlcykgdGhhdCBsaW5rcyB0byB0aGUgZGV2aWNlJ3MKICAgIGRhc2hib2FyZCBwYWdlLiAgU2NhbiBpdCDihpIgb3BlbnMgbGl2ZSB0ZWxlbWV0cnkgaW4gYSBicm93c2VyLgogICAgIiIiCiAgICB1cmwgPSBmIntkYXNoYm9hcmRfYmFzZV91cmx9P2RldmljZT17ZGV2aWNlX2lkfSIKICAgIHFyID0gcXJjb2RlLlFSQ29kZSgKICAgICAgICB2ZXJzaW9uPTEsCiAgICAgICAgZXJyb3JfY29ycmVjdGlvbj1xcmNvZGUuY29uc3RhbnRzLkVSUk9SX0NPUlJFQ1RfTSwKICAgICAgICBib3hfc2l6ZT04LAogICAgICAgIGJvcmRlcj0zLAogICAgKQogICAgcXIuYWRkX2RhdGEodXJsKQogICAgcXIubWFrZShmaXQ9VHJ1ZSkKICAgIGltZyA9IHFyLm1ha2VfaW1hZ2UoZmlsbF9jb2xvcj0iIzFhMWEyZSIsIGJhY2tfY29sb3I9IndoaXRlIikKICAgIGJ1ZiA9IGlvLkJ5dGVzSU8oKQogICAgaW1nLnNhdmUoYnVmLCBmb3JtYXQ9IlBORyIpCiAgICByZXR1cm4gYnVmLmdldHZhbHVlKCkKCgpkZWYgZ2VuZXJhdGVfcXJfYmFzZTY0KGRldmljZV9pZDogc3RyLCBkYXNoYm9hcmRfYmFzZV91cmw6IHN0cikgLT4gc3RyOgogICAgIiIiUmV0dXJuIFFSIGFzIGJhc2U2NCBzdHJpbmcgZm9yIGVtYmVkZGluZyBpbiBIVE1ML1N0cmVhbWxpdC4iIiIKICAgIGltcG9ydCBiYXNlNjQKICAgIHBuZ19ieXRlcyA9IGdlbmVyYXRlX2RldmljZV9xcihkZXZpY2VfaWQsIGRhc2hib2FyZF9iYXNlX3VybCkKICAgIHJldHVybiBiYXNlNjQuYjY0ZW5jb2RlKHBuZ19ieXRlcykuZGVjb2RlKCJ1dGYtOCIpCgoKIyDilIDilIAgUERGIEluY2lkZW50IFJlcG9ydCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKClNFVkVSSVRZX0NPTE9SUyA9IHsKICAgICJjcml0aWNhbCI6IGNvbG9ycy5IZXhDb2xvcigiI2Q2MjcyOCIpLAogICAgImhpZ2giOiAgICAgY29sb3JzLkhleENvbG9yKCIjZmY3ZjBlIiksCiAgICAibWVkaXVtIjogICBjb2xvcnMuSGV4Q29sb3IoIiNmZmJiMzMiKSwKICAgICJsb3ciOiAgICAgIGNvbG9ycy5IZXhDb2xvcigiIzJjYTAyYyIpLAp9CgoKZGVmIGdlbmVyYXRlX2luY2lkZW50X3JlcG9ydCgKICAgIGluY2lkZW50czogTGlzdFtEaWN0XSwKICAgIGFsZXJ0czogTGlzdFtEaWN0XSwKICAgIGFnZW50X2xvZ3M6IExpc3RbRGljdF0sCiAgICBzdGF0czogRGljdCwKICAgIHJ1bl90aW1lc3RhbXA6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAopIC0+IGJ5dGVzOgogICAgIiIiCiAgICBHZW5lcmF0ZSBhIHByb2Zlc3Npb25hbCBBNCBQREYgaW5jaWRlbnQgcmVwb3J0LgogICAgUmV0dXJucyByYXcgUERGIGJ5dGVzIOKAlCBjYWxsZXIgd3JpdGVzIHRvIGZpbGUgb3Igc3RyZWFtcyB0byBkb3dubG9hZC4KICAgICIiIgogICAgYnVmID0gaW8uQnl0ZXNJTygpCiAgICBkb2MgPSBTaW1wbGVEb2NUZW1wbGF0ZSgKICAgICAgICBidWYsCiAgICAgICAgcGFnZXNpemU9QTQsCiAgICAgICAgbGVmdE1hcmdpbj0yMCptbSwgcmlnaHRNYXJnaW49MjAqbW0sCiAgICAgICAgdG9wTWFyZ2luPTE4Km1tLCBib3R0b21NYXJnaW49MTgqbW0sCiAgICApCgogICAgc3R5bGVzID0gZ2V0U2FtcGxlU3R5bGVTaGVldCgpCiAgICB0aXRsZV9zdHlsZSA9IFBhcmFncmFwaFN0eWxlKAogICAgICAgICJUaXRsZSIsIHBhcmVudD1zdHlsZXNbIlRpdGxlIl0sCiAgICAgICAgZm9udFNpemU9MjAsIHRleHRDb2xvcj1jb2xvcnMuSGV4Q29sb3IoIiMxYTFhMmUiKSwKICAgICAgICBzcGFjZUFmdGVyPTQsIGFsaWdubWVudD1UQV9DRU5URVIsCiAgICApCiAgICBoZWFkaW5nX3N0eWxlID0gUGFyYWdyYXBoU3R5bGUoCiAgICAgICAgIkhlYWRpbmciLCBwYXJlbnQ9c3R5bGVzWyJIZWFkaW5nMiJdLAogICAgICAgIGZvbnRTaXplPTEzLCB0ZXh0Q29sb3I9Y29sb3JzLkhleENvbG9yKCIjMWExYTJlIiksCiAgICAgICAgc3BhY2VCZWZvcmU9MTAsIHNwYWNlQWZ0ZXI9NCwKICAgICkKICAgIGJvZHkgPSBzdHlsZXNbIk5vcm1hbCJdCiAgICBib2R5LmZvbnRTaXplID0gOQoKICAgIHRzID0gcnVuX3RpbWVzdGFtcCBvciBkYXRldGltZS51dGNub3coKS5zdHJmdGltZSgiJVktJW0tJWQgJUg6JU0gVVRDIikKICAgIHN0b3J5ID0gW10KCiAgICAjIOKUgOKUgCBDb3ZlciDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0b3J5LmFwcGVuZChTcGFjZXIoMSwgMTAqbW0pKQogICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgi8J+bsO+4jyBBSU9wcyBJb1QgTW9uaXRvcmluZyIsIHRpdGxlX3N0eWxlKSkKICAgIHN0b3J5LmFwcGVuZChQYXJhZ3JhcGgoIkF1dG9tYXRlZCBJbmNpZGVudCBSZXBvcnQiLCBzdHlsZXNbIkhlYWRpbmczIl0pKQogICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaChmIkdlbmVyYXRlZDoge3RzfSIsIGJvZHkpKQogICAgc3RvcnkuYXBwZW5kKEhSRmxvd2FibGUod2lkdGg9IjEwMCUiLCB0aGlja25lc3M9MSwgY29sb3I9Y29sb3JzLkhleENvbG9yKCIjMWExYTJlIikpKQogICAgc3RvcnkuYXBwZW5kKFNwYWNlcigxLCA2Km1tKSkKCiAgICAjIOKUgOKUgCBFeGVjdXRpdmUgc3VtbWFyeSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0b3J5LmFwcGVuZChQYXJhZ3JhcGgoIkV4ZWN1dGl2ZSBTdW1tYXJ5IiwgaGVhZGluZ19zdHlsZSkpCiAgICBvcGVuX2luYyAgPSBzdW0oMSBmb3IgaSBpbiBpbmNpZGVudHMgaWYgaS5nZXQoInN0YXR1cyIpID09ICJvcGVuIikKICAgIGNyaXRpY2FsICA9IHN1bSgxIGZvciBpIGluIGluY2lkZW50cyBpZiBpLmdldCgic2V2ZXJpdHkiKSA9PSAiY3JpdGljYWwiKQogICAgc2VjdXJpdHkgID0gc3VtKDEgZm9yIGkgaW4gaW5jaWRlbnRzIGlmIGkuZ2V0KCJjbGFzc2lmaWNhdGlvbiIpID09ICJzZWN1cml0eSIpCiAgICBzdW1tYXJ5X2RhdGEgPSBbCiAgICAgICAgWyJNZXRyaWMiLCAiVmFsdWUiXSwKICAgICAgICBbIlRvdGFsIEluY2lkZW50cyIsIHN0cihsZW4oaW5jaWRlbnRzKSldLAogICAgICAgIFsiT3BlbiBJbmNpZGVudHMiLCAgc3RyKG9wZW5faW5jKV0sCiAgICAgICAgWyJDcml0aWNhbCBTZXZlcml0eSIsIHN0cihjcml0aWNhbCldLAogICAgICAgIFsiU2VjdXJpdHkgQ2xhc3MuIiwgc3RyKHNlY3VyaXR5KV0sCiAgICAgICAgWyJUb3RhbCBBbGVydHMiLCAgIHN0cihsZW4oYWxlcnRzKSldLAogICAgICAgIFsiRGV2aWNlcyBNb25pdG9yZWQiLCBzdHIoc3RhdHMuZ2V0KCJkZXZpY2VzX3RvdGFsIiwgIuKAlCIpKV0sCiAgICAgICAgWyJSZWFkaW5ncyAoMjRoKSIsICAgIHN0cihzdGF0cy5nZXQoInJlYWRpbmdzXzI0aCIsICLigJQiKSldLAogICAgICAgIFsiQWdlbnQgUnVucyAoMjRoKSIsICBzdHIoc3RhdHMuZ2V0KCJhZ2VudF9ydW5zXzI0aCIsICLigJQiKSldLAogICAgXQogICAgdCA9IFRhYmxlKHN1bW1hcnlfZGF0YSwgY29sV2lkdGhzPVs4MCptbSwgNjAqbW1dKQogICAgdC5zZXRTdHlsZShUYWJsZVN0eWxlKFsKICAgICAgICAoIkJBQ0tHUk9VTkQiLCAoMCwwKSwgKC0xLDApLCBjb2xvcnMuSGV4Q29sb3IoIiMxYTFhMmUiKSksCiAgICAgICAgKCJURVhUQ09MT1IiLCAgKDAsMCksICgtMSwwKSwgY29sb3JzLndoaXRlKSwKICAgICAgICAoIkZPTlROQU1FIiwgICAoMCwwKSwgKC0xLDApLCAiSGVsdmV0aWNhLUJvbGQiKSwKICAgICAgICAoIkZPTlRTSVpFIiwgICAoMCwwKSwgKC0xLC0xKSwgOSksCiAgICAgICAgKCJST1dCQUNLR1JPVU5EUyIsICgwLDEpLCAoLTEsLTEpLCBbY29sb3JzLkhleENvbG9yKCIjZjVmNWY1IiksIGNvbG9ycy53aGl0ZV0pLAogICAgICAgICgiR1JJRCIsICAgICAgICgwLDApLCAoLTEsLTEpLCAwLjQsIGNvbG9ycy5IZXhDb2xvcigiI2NjY2NjYyIpKSwKICAgICAgICAoIkxFRlRQQURESU5HIiwgKDAsMCksICgtMSwtMSksIDYpLAogICAgICAgICgiUklHSFRQQURESU5HIiwgKDAsMCksICgtMSwtMSksIDYpLAogICAgICAgICgiVE9QUEFERElORyIsICAgKDAsMCksICgtMSwtMSksIDQpLAogICAgICAgICgiQk9UVE9NUEFERElORyIsKDAsMCksICgtMSwtMSksIDQpLAogICAgXSkpCiAgICBzdG9yeS5hcHBlbmQodCkKICAgIHN0b3J5LmFwcGVuZChTcGFjZXIoMSwgNiptbSkpCgogICAgIyDilIDilIAgSW5jaWRlbnRzIHRhYmxlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgiQ2xhc3NpZmllZCBJbmNpZGVudHMiLCBoZWFkaW5nX3N0eWxlKSkKICAgIGlmIGluY2lkZW50czoKICAgICAgICBpbmNfZGF0YSA9IFtbIiMiLCAiRGV2aWNlIiwgIlRocmVhdCBUeXBlIiwgIkNsYXNzLiIsICJTZXZlcml0eSIsICJTdGF0dXMiXV0KICAgICAgICBmb3IgaSwgaW5jIGluIGVudW1lcmF0ZShpbmNpZGVudHNbOjUwXSwgMSk6CiAgICAgICAgICAgIGluY19kYXRhLmFwcGVuZChbCiAgICAgICAgICAgICAgICBzdHIoaSksCiAgICAgICAgICAgICAgICBpbmMuZ2V0KCJkZXZpY2VfaWQiLCAiIiksCiAgICAgICAgICAgICAgICBpbmMuZ2V0KCJ0aHJlYXRfdHlwZSIsICIiKSwKICAgICAgICAgICAgICAgIGluYy5nZXQoImNsYXNzaWZpY2F0aW9uIiwgIiIpLAogICAgICAgICAgICAgICAgaW5jLmdldCgic2V2ZXJpdHkiLCAiIiksCiAgICAgICAgICAgICAgICBpbmMuZ2V0KCJzdGF0dXMiLCAiIiksCiAgICAgICAgICAgIF0pCiAgICAgICAgdDIgPSBUYWJsZShpbmNfZGF0YSwgY29sV2lkdGhzPVsxMCptbSwgMzUqbW0sIDQwKm1tLCAyOCptbSwgMjIqbW0sIDIyKm1tXSkKICAgICAgICB0czIgPSBUYWJsZVN0eWxlKFsKICAgICAgICAgICAgKCJCQUNLR1JPVU5EIiwgKDAsMCksICgtMSwwKSwgY29sb3JzLkhleENvbG9yKCIjMWExYTJlIikpLAogICAgICAgICAgICAoIlRFWFRDT0xPUiIsICAoMCwwKSwgKC0xLDApLCBjb2xvcnMud2hpdGUpLAogICAgICAgICAgICAoIkZPTlROQU1FIiwgICAoMCwwKSwgKC0xLDApLCAiSGVsdmV0aWNhLUJvbGQiKSwKICAgICAgICAgICAgKCJGT05UU0laRSIsICAgKDAsMCksICgtMSwtMSksIDgpLAogICAgICAgICAgICAoIkdSSUQiLCAgICAgICAoMCwwKSwgKC0xLC0xKSwgMC40LCBjb2xvcnMuSGV4Q29sb3IoIiNjY2NjY2MiKSksCiAgICAgICAgICAgICgiUk9XQkFDS0dST1VORFMiLCAoMCwxKSwgKC0xLC0xKSwgW2NvbG9ycy5IZXhDb2xvcigiI2Y5ZjlmOSIpLCBjb2xvcnMud2hpdGVdKSwKICAgICAgICAgICAgKCJMRUZUUEFERElORyIsICgwLDApLCAoLTEsLTEpLCA1KSwKICAgICAgICAgICAgKCJUT1BQQURESU5HIiwgICgwLDApLCAoLTEsLTEpLCAzKSwKICAgICAgICAgICAgKCJCT1RUT01QQURESU5HIiwoMCwwKSwoLTEsLTEpLCAzKSwKICAgICAgICBdKQogICAgICAgICMgQ29sb3VyIHNldmVyaXR5IGNlbGxzCiAgICAgICAgZm9yIHJvd19pZHgsIGluYyBpbiBlbnVtZXJhdGUoaW5jaWRlbnRzWzo1MF0sIDEpOgogICAgICAgICAgICBzZXYgPSBpbmMuZ2V0KCJzZXZlcml0eSIsICJsb3ciKQogICAgICAgICAgICBiZyA9IFNFVkVSSVRZX0NPTE9SUy5nZXQoc2V2LCBjb2xvcnMud2hpdGUpCiAgICAgICAgICAgIHRzMi5hZGQoIkJBQ0tHUk9VTkQiLCAoNCwgcm93X2lkeCksICg0LCByb3dfaWR4KSwgYmcpCiAgICAgICAgICAgIHRzMi5hZGQoIlRFWFRDT0xPUiIsICAoNCwgcm93X2lkeCksICg0LCByb3dfaWR4KSwgY29sb3JzLndoaXRlKQogICAgICAgIHQyLnNldFN0eWxlKHRzMikKICAgICAgICBzdG9yeS5hcHBlbmQodDIpCiAgICBlbHNlOgogICAgICAgIHN0b3J5LmFwcGVuZChQYXJhZ3JhcGgoIk5vIGluY2lkZW50cyByZWNvcmRlZCBpbiB0aGlzIHJ1bi4iLCBib2R5KSkKICAgIHN0b3J5LmFwcGVuZChTcGFjZXIoMSwgNiptbSkpCgogICAgIyDilIDilIAgUmVjb21tZW5kYXRpb25zIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgcmVjcyA9IFtpIGZvciBpIGluIGluY2lkZW50cyBpZiBpLmdldCgicmVjb21tZW5kYXRpb24iKV0KICAgIGlmIHJlY3M6CiAgICAgICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgiQWdlbnQgUmVtZWRpYXRpb24gUmVjb21tZW5kYXRpb25zIiwgaGVhZGluZ19zdHlsZSkpCiAgICAgICAgZm9yIGluYyBpbiByZWNzWzoyMF06CiAgICAgICAgICAgIHNldiAgID0gaW5jLmdldCgic2V2ZXJpdHkiLCAibG93IikKICAgICAgICAgICAgZW1vamkgPSB7ImNyaXRpY2FsIjoi8J+UtCIsImhpZ2giOiLwn5+gIiwibWVkaXVtIjoi8J+foSIsImxvdyI6IvCfn6IifS5nZXQoc2V2LCLimqoiKQogICAgICAgICAgICBzdG9yeS5hcHBlbmQoUGFyYWdyYXBoKAogICAgICAgICAgICAgICAgZiI8Yj57ZW1vaml9IHtpbmMuZ2V0KCdkZXZpY2VfaWQnKX0g4oCUIHtpbmMuZ2V0KCd0aHJlYXRfdHlwZScpfTwvYj4iLAogICAgICAgICAgICAgICAgYm9keSwKICAgICAgICAgICAgKSkKICAgICAgICAgICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgKICAgICAgICAgICAgICAgIGYiJm5ic3A7Jm5ic3A7Jm5ic3A7e2luYy5nZXQoJ3JlY29tbWVuZGF0aW9uJywgJycpfSIsCiAgICAgICAgICAgICAgICBib2R5LAogICAgICAgICAgICApKQogICAgICAgICAgICBzdG9yeS5hcHBlbmQoU3BhY2VyKDEsIDIqbW0pKQogICAgICAgIHN0b3J5LmFwcGVuZChTcGFjZXIoMSwgNCptbSkpCgogICAgIyDilIDilIAgQWdlbnQgYXVkaXQgdHJhaWwg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBzdG9yeS5hcHBlbmQoUGFyYWdyYXBoKCJBZ2VudCBBdWRpdCBUcmFpbCAobGFzdCAyMCBlbnRyaWVzKSIsIGhlYWRpbmdfc3R5bGUpKQogICAgaWYgYWdlbnRfbG9nczoKICAgICAgICBsb2dfZGF0YSA9IFtbIkFnZW50IiwgIlRhc2siLCAiVmFsaWRhdGlvbiIsICJUaW1lIChtcykiXV0KICAgICAgICBmb3IgbG9nIGluIGFnZW50X2xvZ3NbOjIwXToKICAgICAgICAgICAgbG9nX2RhdGEuYXBwZW5kKFsKICAgICAgICAgICAgICAgIGxvZy5nZXQoImFnZW50X25hbWUiLCAiIiksCiAgICAgICAgICAgICAgICBsb2cuZ2V0KCJ0YXNrIiwgIiIpLAogICAgICAgICAgICAgICAgbG9nLmdldCgidmFsaWRhdGlvbl9zdGF0dXMiLCAiIiksCiAgICAgICAgICAgICAgICBzdHIobG9nLmdldCgiZXhlY3V0aW9uX3RpbWVfbXMiLCAwKSksCiAgICAgICAgICAgIF0pCiAgICAgICAgdDMgPSBUYWJsZShsb2dfZGF0YSwgY29sV2lkdGhzPVs0NSptbSwgNTUqbW0sIDQwKm1tLCAyNSptbV0pCiAgICAgICAgdDMuc2V0U3R5bGUoVGFibGVTdHlsZShbCiAgICAgICAgICAgICgiQkFDS0dST1VORCIsICgwLDApLCAoLTEsMCksIGNvbG9ycy5IZXhDb2xvcigiIzFhMWEyZSIpKSwKICAgICAgICAgICAgKCJURVhUQ09MT1IiLCAgKDAsMCksICgtMSwwKSwgY29sb3JzLndoaXRlKSwKICAgICAgICAgICAgKCJGT05UTkFNRSIsICAgKDAsMCksICgtMSwwKSwgIkhlbHZldGljYS1Cb2xkIiksCiAgICAgICAgICAgICgiRk9OVFNJWkUiLCAgICgwLDApLCAoLTEsLTEpLCA4KSwKICAgICAgICAgICAgKCJHUklEIiwgICAgICAgKDAsMCksICgtMSwtMSksIDAuNCwgY29sb3JzLkhleENvbG9yKCIjY2NjY2NjIikpLAogICAgICAgICAgICAoIlJPV0JBQ0tHUk9VTkRTIiwoMCwxKSwoLTEsLTEpLFtjb2xvcnMuSGV4Q29sb3IoIiNmOWY5ZjkiKSxjb2xvcnMud2hpdGVdKSwKICAgICAgICAgICAgKCJMRUZUUEFERElORyIsKDAsMCksKC0xLC0xKSw1KSwKICAgICAgICAgICAgKCJUT1BQQURESU5HIiwgKDAsMCksKC0xLC0xKSwzKSwKICAgICAgICAgICAgKCJCT1RUT01QQURESU5HIiwoMCwwKSwoLTEsLTEpLDMpLAogICAgICAgIF0pKQogICAgICAgIHN0b3J5LmFwcGVuZCh0MykKICAgIGVsc2U6CiAgICAgICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgiTm8gYWdlbnQgbG9ncyBhdmFpbGFibGUuIiwgYm9keSkpCgogICAgIyDilIDilIAgRm9vdGVyIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgc3RvcnkuYXBwZW5kKFNwYWNlcigxLCA4Km1tKSkKICAgIHN0b3J5LmFwcGVuZChIUkZsb3dhYmxlKHdpZHRoPSIxMDAlIiwgdGhpY2tuZXNzPTAuNSwgY29sb3I9Y29sb3JzLkhleENvbG9yKCIjY2NjY2NjIikpKQogICAgc3RvcnkuYXBwZW5kKFBhcmFncmFwaCgKICAgICAgICAiR2VuZXJhdGVkIGJ5IEFJT3BzIElvVCBNb25pdG9yaW5nIFBsYXRmb3JtIMK3IDctQWdlbnQgQ3Jld0FJIFBpcGVsaW5lIMK3ICIKICAgICAgICAiSXNvbGF0aW9uRm9yZXN0IEFub21hbHkgRGV0ZWN0aW9uIMK3IElkZW50aXR5LUF3YXJlIEFnZW50cyIsCiAgICAgICAgUGFyYWdyYXBoU3R5bGUoImZvb3RlciIsIHBhcmVudD1ib2R5LCBmb250U2l6ZT03LAogICAgICAgICAgICAgICAgICAgICAgIHRleHRDb2xvcj1jb2xvcnMuSGV4Q29sb3IoIiM4ODg4ODgiKSwgYWxpZ25tZW50PVRBX0NFTlRFUiksCiAgICApKQoKICAgIGRvYy5idWlsZChzdG9yeSkKICAgIHJldHVybiBidWYuZ2V0dmFsdWUoKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICAjIFNtb2tlIHRlc3QKICAgIHByaW50KCJbUVJdIEdlbmVyYXRpbmcgZGV2aWNlIFFSLi4uIikKICAgIHBuZyA9IGdlbmVyYXRlX2RldmljZV9xcigiSU9ULTEwMDAiLCAiaHR0cDovL2xvY2FsaG9zdDo4NTAxIikKICAgIHByaW50KGYiW1FSXSBQTkcgYnl0ZXM6IHtsZW4ocG5nKX0iKQoKICAgIHByaW50KCJbUERGXSBHZW5lcmF0aW5nIGluY2lkZW50IHJlcG9ydC4uLiIpCiAgICBkdW1teV9pbmNpZGVudHMgPSBbCiAgICAgICAgeyJkZXZpY2VfaWQiOiJJT1QtMTAwMCIsInRocmVhdF90eXBlIjoidGVtcF9zcGlrZSIsImNsYXNzaWZpY2F0aW9uIjoiZW52aXJvbm1lbnRhbCIsCiAgICAgICAgICJzZXZlcml0eSI6ImhpZ2giLCJzdGF0dXMiOiJvcGVuIiwicmVjb21tZW5kYXRpb24iOiJDaGVjayBIVkFDIGluIFpvbmUgQSJ9LAogICAgICAgIHsiZGV2aWNlX2lkIjoiSU9ULTEwMDMiLCJ0aHJlYXRfdHlwZSI6InRhbXBlcmluZyIsImNsYXNzaWZpY2F0aW9uIjoic2VjdXJpdHkiLAogICAgICAgICAic2V2ZXJpdHkiOiJjcml0aWNhbCIsInN0YXR1cyI6Im9wZW4iLCJyZWNvbW1lbmRhdGlvbiI6IlF1YXJhbnRpbmUgZGV2aWNlIGltbWVkaWF0ZWx5In0sCiAgICBdCiAgICBkdW1teV9hbGVydHMgPSBbeyJkZXZpY2VfaWQiOiJJT1QtMTAwMCIsInNldmVyaXR5IjoiaGlnaCIsImFsZXJ0X3R5cGUiOiJ0ZW1wX3NwaWtlIn1dICogNQogICAgZHVtbXlfbG9ncyA9IFt7ImFnZW50X25hbWUiOiJhbm9tYWx5X2RldGVjdG9yIiwidGFzayI6InNjYW4iLCJ2YWxpZGF0aW9uX3N0YXR1cyI6InZhbGlkYXRlZCIsImV4ZWN1dGlvbl90aW1lX21zIjoyMTB9XQogICAgZHVtbXlfc3RhdHMgPSB7ImRldmljZXNfdG90YWwiOjEyLCJyZWFkaW5nc18yNGgiOjEyMDAsImFnZW50X3J1bnNfMjRoIjozfQogICAgcGRmX2J5dGVzID0gZ2VuZXJhdGVfaW5jaWRlbnRfcmVwb3J0KGR1bW15X2luY2lkZW50cywgZHVtbXlfYWxlcnRzLCBkdW1teV9sb2dzLCBkdW1teV9zdGF0cykKICAgIHByaW50KGYiW1BERl0gUERGIGJ5dGVzOiB7bGVuKHBkZl9ieXRlcyl9IikKICAgIHdpdGggb3BlbigiL3RtcC90ZXN0X3JlcG9ydC5wZGYiLCAid2IiKSBhcyBmOgogICAgICAgIGYud3JpdGUocGRmX2J5dGVzKQogICAgcHJpbnQoIltQREZdIFNhdmVkIHRvIC90bXAvdGVzdF9yZXBvcnQucGRmIikK"}')
for path, b64 in BUNDLE.items():
    d = os.path.dirname(path)
    if d: os.makedirs(d, exist_ok=True)
    with open(path, 'w') as f:
        f.write(base64.b64decode(b64).decode() if b64 else '')
print(f'Unpacked {len(BUNDLE)} source files')
for p in BUNDLE: print(f'  {p}')

In [ ]:
# Cell 3: Environment
import os
os.environ.update({
    'DATABASE_URL': 'sqlite:///./aiops_demo.db',
    'JWT_SECRET': 'colab-demo-secret',
    'LLM_PROVIDER': 'ollama',
    'ADMIN_USERNAME': 'admin',
    'ADMIN_PASSWORD': 'admin123',
})
print('Environment configured')

In [ ]:
# Cell 4: Init DB
from backend.db.models import init_db, SessionLocal, Device, SensorReading, Alert, Incident, AgentTaskLog
from backend.db.models import engine
from sqlalchemy import inspect as sa_inspect
init_db()
tables = sa_inspect(engine).get_table_names()
print(f'Database ready — {len(tables)} tables: {tables}')

In [ ]:
# Cell 5: Seed simulator
from backend.simulator.sensor_sim import generate_devices, generate_batch
from datetime import datetime
db = SessionLocal()
devices = generate_devices(12)
for d in devices:
    if not db.query(Device).filter(Device.device_id == d['device_id']).first():
        db.add(Device(**d))
db.commit()
readings = generate_batch(devices, readings_per_device=100, anomaly_rate=0.08)
for r in readings:
    db.add(SensorReading(
        device_id=r['device_id'], timestamp=datetime.fromisoformat(r['timestamp']),
        temperature=r['temperature'], humidity=r['humidity'],
        vibration=r['vibration'], battery=r['battery'], signal_strength=r['signal_strength']
    ))
db.commit()
print(f'{db.query(Device).count()} devices, {db.query(SensorReading).count()} readings seeded')
db.close()

In [ ]:
# Cell 6: Train IsolationForest
from backend.ml.anomaly_detector import AnomalyDetector, classify_severity, classify_alert_type
db = SessionLocal()
rd = [{'temperature':r.temperature,'humidity':r.humidity,'vibration':r.vibration,
       'battery':r.battery,'signal_strength':r.signal_strength}
      for r in db.query(SensorReading).all()]
db.close()
det = AnomalyDetector(contamination=0.08)
stats = det.fit(rd)
det.save()
print('IsolationForest trained')
for k,v in stats.items(): print(f'  {k}: {v}')

In [ ]:
# Cell 7: Anomaly detection + write alerts
db = SessionLocal()
det2 = AnomalyDetector()
det2.load()
written = 0
for r in db.query(SensorReading).limit(300).all():
    rd = {'temperature':r.temperature,'humidity':r.humidity,'vibration':r.vibration,
          'battery':r.battery,'signal_strength':r.signal_strength}
    is_anom, score = det2.predict(rd)
    if is_anom:
        db.add(Alert(device_id=r.device_id, severity=classify_severity(score),
                     alert_type=classify_alert_type(rd),
                     description=f'Score {score:.3f} temp={r.temperature:.1f}C',
                     agent_source='anomaly_detector', anomaly_flag=True))
        written += 1
db.commit()
from collections import Counter
sevs = Counter(a.severity for a in db.query(Alert).all())
db.close()
print(f'{written} alerts: {dict(sevs)}')

In [ ]:
# Cell 8: Identity-aware agent tokens
from backend.auth.identity import issue_agent_token, verify_agent_token, has_permission
agents = ['telemetry_ingestion','device_health','anomaly_detector',
          'security','incident_classifier','response_recommender','validator']
print(f'{"Agent":<25} {"Permissions"}')
print('-'*55)
for name in agents:
    tok = issue_agent_token(name)
    c = verify_agent_token(tok)
    print(f"{name:<25} {', '.join(c['permissions'])}")
print()
sec = issue_agent_token('security')
for p in ['write:incidents','write:alerts','write:devices']:
    print(f"  security → {p}: {'OK' if has_permission(sec,p) else 'DENIED'}")

In [ ]:
# Cell 9: Device QR code
from backend.outputs import generate_device_qr
from IPython.display import display, Image
db = SessionLocal()
device = db.query(Device).first()
db.close()
png = generate_device_qr(device.device_id, 'http://localhost:8501')
display(Image(data=png, width=180))
print(f'QR for {device.device_id} @ {device.location} (battery {device.battery_level}%)')

In [ ]:
# Cell 10: PDF incident report
from backend.outputs import generate_incident_report
from google.colab import files
db = SessionLocal()
incs  = [{'device_id':i.device_id,'threat_type':i.threat_type,'classification':i.classification,
           'severity':i.severity,'status':i.status,'recommendation':i.recommendation}
         for i in db.query(Incident).limit(50).all()]
alrts = [{'device_id':a.device_id,'severity':a.severity,'alert_type':a.alert_type}
         for a in db.query(Alert).limit(100).all()]
logs  = [{'agent_name':l.agent_name,'task':l.task,'validation_status':l.validation_status,
           'execution_time_ms':l.execution_time_ms} for l in db.query(AgentTaskLog).limit(20).all()]
stats = {'devices_total':db.query(Device).count(),
         'readings_24h':db.query(SensorReading).count(),'agent_runs_24h':0}
db.close()
pdf = generate_incident_report(incs, alrts, logs, stats)
with open('aiops_report.pdf','wb') as f: f.write(pdf)
print(f'PDF generated: {len(pdf):,} bytes')
files.download('aiops_report.pdf')

In [ ]:
# Cell 11: Summary
db = SessionLocal()
from collections import Counter
print('='*55)
print('AIOps IoT Demo Complete')
print('='*55)
print(f'  Devices:   {db.query(Device).count()}')
print(f'  Readings:  {db.query(SensorReading).count()}')
print(f'  Alerts:    {db.query(Alert).count()}')
print(f'  Incidents: {db.query(Incident).count()}')
types = Counter(a.alert_type for a in db.query(Alert).all())
print('\nTop alert types:')
for t,c in types.most_common(5): print(f'  {t:<25} {c}')
db.close()